# [2.1] - Intro to RL (exercises)

> **ARENA [Course Page](https://learn.arena.education/chapter2_rl/01_intro_rl/intro/)**
>
> **Colab: [exercises](https://colab.research.google.com/github/ARENA-education/ARENA_materials/blob/main/chapter2_rl/exercises/part1_intro_to_rl/2.1_Intro_to_RL_exercises.ipynb?t=20261108) | [solutions](https://colab.research.google.com/github/ARENA-education/ARENA_materials/blob/main/chapter2_rl/exercises/part1_intro_to_rl/2.1_Intro_to_RL_solutions.ipynb?t=20261108)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://info-arena.github.io/ARENA_img/slack.html), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://learn.arena.education/chapter0_fundamentals/), [(1) Transformer Interpretability](https://learn.arena.education/chapter1_transformer_interp/), [(2) RL](https://learn.arena.education/chapter2_rl/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-21.png" width="350">

# Introduction

This section is designed to bring you up to speed with the basics of reinforcement learning. Before we cover the big topics like DQN, VPG, PPO and RLHF, we need to build a strong theoretical foundation first.

For now, we will assume the environment has a finite set of states/actions, and we explore two cases:

1. that the dynamics of the environment are directly accessible to the agent, so that we can solve for an optimal policy *analytically*. 
2. the environment is a black box that we can sample from, and the agent must learn from play how the environment works. 

In the DQN section, we weaken the assumption that there are few enough states we could visit them all, and instead build something with deep learning that can generalize to never-before-seen states.

This day's material will be more theory heavy, as the goal is to understand the fundamentals of RL before we apply deep learning to it.

For a lecture on the material today, which provides some high-level understanding before you dive into the material, watch the video below:

<iframe width="540" height="304" src="https://www.youtube.com/embed/Q2cFcq3I0G8" frameborder="0" referrerpolicy="strict-origin-when-cross-origin" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

## Content & Learning Objectives

### 1️⃣ Planning

> ##### Learning Objectives
> * Understand Markov Decision Processes and the goal of the agent, for known environments.
> * Understand the Bellman equation.
> * Understand the policy improvement theorem, and how we can use it to iteratively solve for an optimal policy.

### 2️⃣ Learning

* **SARSA** and **Q-learning** are methods of learning in Markovian domains by experiencing the consequences of actions, and learning from them, without peeking directly at how the environment works. The agent must learn the dynamic of the environment solely from experience.

* We can do short updates and bootstrap the rest, or we can do long updates, but suffer higher variance. Eligibility traces allow us to get the best of both worlds in an efficient fashion.

> ##### Learning Objectives
> * Understand the anatomy of a `gym.Env`, so that you feel comfortable using them and writing your own.
> * Understand the ideas behind temporal difference methods such as SARSA and Q-learning.
> * Implement SARSA and Q-Learning, and compare them on different environments.
> * Understand the TD($\lambda$) algorithm, and how it can we used to mix over short and long timescale updates.

### 3️⃣ (Optional) Multi-Armed Bandit

Multi-armed bandits are a different flavour of environment where there are no states, but the environmental distribution itself may drift over time. Here, the emphasis is on strategies to trade-off exploration v.s. exploitation, and methods from bandits (such as UCB) are used elsewhere 
(e.g [Monte Carlo Tree Search](https://en.wikipedia.org/wiki/Monte_Carlo_tree_search), or the [Hyperband algorithm](https://arxiv.org/abs/1603.06560) for hyperparameter search). 

This used to be core material, but as none of the later material requires understanding bandits, we've moved it to bonus optional material.

> ##### Learning Objectives
> * Understand the difficulty of optimal exploration
> * Compare *Upper Confidence Bound* (UCB) methods to other exploration methods.

## Setup code

In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
chapter = "chapter2_rl"

# Locate the repo root (the directory that contains the chapter folder) and work from its exercises dir.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")
os.chdir(f"{root}/{chapter}/exercises")


In [2]:
import os
import random
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import TypeAlias

import einops
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from tqdm.auto import tqdm

Arr: TypeAlias = np.ndarray

max_episode_steps = 1000
N_RUNS = 200

# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part1_intro_to_rl"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part1_intro_to_rl.tests as tests
import part1_intro_to_rl.utils as utils
from plotly_utils import cliffwalk_imshow, imshow, line
from part1_intro_to_rl.utils import GridWorld, Environment

# 1️⃣ Planning

> ##### Learning Objectives
>
> * Understand the Bellman equation.
> * Understand the policy improvement theorem, and how we can use it to iteratively solve for an optimal policy.

## Planning in Known Environments

We are presented with a white-box environment: we have access to the environmental distribution $T$ and reward function $R$. The goal is to turn the problem of finding the best policy $\pi^*$ into an optimization problem, which we can then solve iteratively.

Here, we assume environments small enough that visiting all state-action pairs is tractable. Tabular RL doesn't learn any relationships between states that could be treated similarly, but instead keep track of how valuable each state is in a large lookup table.

## Readings (optional)

There are no compulsory readings before the material in this section (the lecture covers everything), although the following sections of [Sutton and Barto](https://www.andrew.cmu.edu/course/10-703/textbook/BartoSutton.pdf) are relevant and so you may want to refer to them if you get stuck / confused with anything:

- Chapter 3: Sections 3.1, 3.2, 3.3, 3.4, 3.5, 3.6
- Chapter 4: Sections 4.1, 4.2, 4.3, 4.4

## What is Reinforcement Learning?

In reinforcement learning, the agent interacts with an environment in a loop: The agent in state $s$ uses its current policy $\pi$ to sample an **action** $a \sim \pi(\cdot | s)$ to the environment, and the environment replies with **state, reward** pair $(s',r)$, where the new state is sampled from the environmental transition distribution $s' \sim T(\cdot | s,a)$ conditioned on the previous state $s$ and action $a$, and the reward $r = R(s,a,s')$ is computed using the reward function $R$, given the state,action,next-state triple $(s,a,s')$. This implies both the environment and the policy are **Markovian**, in that their dynamics depend only on the previous state and action (for the environment) or on the current state (for the policy).

**Note** - some of the sections below have a TLDR at the start. This indicates that not the entire section is crucially important to read and understand (e.g. it's a small pedantic note or a mathematical divergence), and you can safely read the TLDR and move onto the next section if you want.

<details>
<summary>Policy vs Agent?</summary>

Technically, the agent is the decision process that chooses the current policy, and the policy itself is just a conditional distribution over actions given the current state.
We're often sloppy and use the terms interchangeably (e.g. "the agent chooses an action"), but it's important to be aware of the distinction.
</details>

### Trajectories

We define a **trajectory** or **rollout** (denoted $\tau$) in RL as the full history of interaction between the agent and the environment:

$$
\tau = s_0, a_0, r_1, s_1, a_1, r_2, s_2, a_2, r_3, \ldots
$$

Under this notation, the timestep increments during the environment's turn to interact. In other words, during timestep $t$, given state $s_t$ the agent returns action $a_t$ (in the same time step), but given $(s_t, a_t)$ the environment generates $(s_{t+1}, r_{t+1})$.

> *Note - some authors use the convention that the agent's action causes the timestep transition, i.e. the trajectory looks like $s_0, a_0, \textcolor{red}{r_0}, s_1, a_1, \textcolor{red}{r_1}, \ldots$. We're following the convention in Sutton & Barto here, but it's important to be aware of this possible notation difference when reading other sources.*

The agent chooses actions using a policy $\pi$, which we can think of as either a deterministic function $a = \pi(s)$ from states to actions, or more generally a stochastic function from which actions are sampled, $a \sim \pi(\cdot | s)$.

Implicitly, we have assumed that the agent need only be Markovian as well (i.e. the action depends on the current state, not past states). Do you think this is a reasonable assumption? You should think about this before reading the dropdown.

<details>
<summary>Answer / discussion</summary>

In most environments, this may well be a reasonable assumption. For example, in a game of tic-tac-toe, knowledge of the current state of the board is sufficient to determine the optimal move, how the board got to that state is irrelevant.

There are more complex environments where a Markovian assumption wouldn't work. One classic example is a game of poker, since past player behaviour (betting patterns, bluffing, etc) might change what the optimal strategy is at a given point in time, even if the resulting game state is the same.

Of course, a non-Markovian game can be turned Markovian by redefining the state space to include relevant historical information, but this is splitting hairs (and it also blows up the state space).
</details>

### Value function

The goal of the agent is to choose a policy that maximizes the **expected discounted return** $G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \ldots$, the expected sum of future discounted rewards it would expect to obtain by following its currently chosen **policy** $\pi$. We call the expected discounted return from a state $s$ following policy $\pi$ the **value function** $V_{\pi}(s)$, defined as
$$
\begin{align*}
V_{\pi}(s) &= \mathbb{E}_{\pi} \left[ G_t \mid s_t = s \right] \\
&= \mathbb{E}_{\pi} \left[ \sum_{i=0}^\infty \gamma^{i} r_{i + t+1} \Bigg| s_t = s \right] \\
&= \mathbb{E}_{\pi} \left[ r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \ldots \mid s_t = s \right]
\end{align*}
$$
where the expectation is with respect to sampling actions from $\pi$, and (implicitly) sampling states and rewards from $T$ and $R$.

Here are a few divergences into the exact nature of the reward function. You can just read the TLDRs and move on if you're not super interested in the deeper mathematics.

<details>
<summary>Divergence #1: why do we discount? <i>TLDR - not discounting leads to possibly weird behaviour with infinite non-converging sums</i></summary>

We would like a way to signal to the agent that reward now is better than reward later.
If we didn't discount, the sum $\sum_{i=t}^\infty r_i$ may diverge.
This leads to strange behavior, as we can't meaningfully compare the returns for sequences of rewards that diverge. Trying to sum the sequence $1,1,1,\ldots$ or $2,2,2,2\ldots$ or even $2, -1, 2 , -1, \ldots$ is a problem since they all diverge to positive infinity. An agent with a policy that leads to infinite expected return might become lazy, as waiting around doing nothing for a thousand years, and then acting optimally leads to the same return (infinite) as playing optimal from the first time step.

Worse still, we can have reward sequences for which the sum may never approach any finite value, nor diverge to $\pm \infty$ (like summing $-1, 1, -1 ,1 , \ldots$).
We could otherwise patch this by requiring that the agent has a finite number of interactions with the environment (this is often true, and is called an **episodic** environment that we will see later) or restrict ourselves to environments for which the expected return for the optimal policy is finite, but these can often be undesirable constraints to place.

</details>

<details>
<summary>Divergence #2: why do we discount geometrically? <i>TLDR - geometric discounting is "time consistent", since the nature of the problem looks the same from each timestep.</i></summary>

In general, we could consider a more general discount function $\Gamma : \mathbb{N} \to [0,1]$ and define the discounted return as $\sum_{i=t}^\infty \Gamma(i) r_i$. The geometric discount $\Gamma(i) = \gamma^i$ is commonly used, but other discounts include the hyperbolic discount $\Gamma(i) = \frac{1}{1 + iD}$, where $D>0$ is a hyperparameter. (Humans are [often said](https://chris-said.io/2018/02/04/hyperbolic-discounting/) to act as if they use a hyperbolic discount.) Other than  being mathematically convenient to work with (as the sum of geometric discounts has an elegant closed form expression $\sum_{i=0}^\infty \gamma^i  = \frac{1}{1-\gamma}$), geometric is preferred as it is *time consistent*, that is, the discount from one time step to the next remains constant.
Rewards at timestep $t+1$ are always worth a factor of $\gamma$ less than rewards on timestep $t$

$$
\frac{\Gamma(t+1)}{\Gamma(t)} = \frac{\gamma^{t+1}}{\gamma^t} = \gamma
$$

whereas for the hyperbolic discount, the amount discounted from one step to the next
is a function of the timestep itself, and decays to 1 (no discount)

$$
\frac{\Gamma(t+1)}{\Gamma(t)} =  \frac{1+tD}{1+(t+1)D} \to 1
$$

so very little discounting is done once rewards are far away enough in the future.

Hence, using geometric discount lends a certain symmetry to the problem. If after $N$ actions the agent ends up in exactly the same state as it was in before, then there will be less total value remaining (because of the decay rate), but the optimal policy won't change (because the rewards the agent gets if it takes a series of actions will just be a scaled-down version of the rewards it would have gotten for those actions $N$ steps ago).

</details>


<details>
<summary> Divergence #3: why do both sides not depend on t? 
<i> TLDR - value function is time invariant. </i> </summary>

The value function does not have explicit time dependence on the left-hand side because the 
underlying process is **stationary** under policy $\pi$ (that is, the environment dynamics $T$ and $R$, and the policy $\pi$, all depend on what state you're in or what action you took, but not when it was done.)
Because of these assumptions, the distribution of future rewards depends only on the **current state** $s$, not on the absolute time $t$. By analogy, the best move in a game of chess depends entirely on the current board state, and not when during the game it happened. If we returned to the same board position twice in a game, the best move from that point is the same as what it was last time it was encountered.

</details>

### Bellman equation

We can write the value function in the following recursive manner, called the **Bellman equation**:

$$
V_\pi(s) = \sum_a \pi(a | s) \sum_{s'} T(s' | s, a) \bigg( R(s, a, s') + \gamma V_\pi(s') \bigg)
$$

(Optional) Try to prove this for yourself!

<details>
<summary>Hint</summary>

Start by writing the expectation as a sum over all possible actions $a$ that can be taken from state $s$. Then, try to separate $r_{t+1}$ and the later reward terms inside the sum.
</details>

<details>
<summary>Answer</summary>

Write this as a sum over actions according to the policy $\pi$:

$$
V_\pi(s) = \sum_a \pi(a \mid s) \; \mathbb{E}_\pi[G_t \mid s_t = s, a_t = a]
$$

Separate out the first reward $r_{t+1}$:

$$
V_\pi(s) = \sum_a \pi(a \mid s) \; \mathbb{E}_\pi \Big[ r_{t+1} + \gamma G_{t+1} \;\mid\; s_t = s, a_t = a \Big]
$$

Expand over next states $s'$ using the transition probability $T(s' \mid s, a)$ and reward function $R(s, a, s')$:

$$
V_\pi(s) = \sum_a \pi(a \mid s) \sum_{s'} T(s' \mid s, a) \Big[ R(s, a, s') + \gamma \; \mathbb{E}_\pi[G_{t+1} \mid s_{t+1} = s'] \Big]
$$

Recognize that $\mathbb{E}_\pi[G_{t+1} \mid s_{t+1} = s'] = V_\pi(s')$, giving the **Bellman equation**:

$$
V_\pi(s) = \sum_a \pi(a \mid s) \sum_{s'} T(s' \mid s, a) \big[ R(s, a, s') + \gamma V_\pi(s') \big]
$$

How should we interpret this? This formula tells us that the total value from present time can be written as sum of next-timestep rewards and value terms discounted by a factor of $\gamma$. Recall earlier in our discussion of **geometric** vs **hyperbolic** discounting, we argued that geometric discounting has a symmetry through time, because of the constant discount factor. This is exactly what this formula shows, just on the scale of a single step. If we had used a non-geometric discount, we wouldn't be able to get this nice recursive relationship (try and see why!)

</details>

The Bellman equation can be thought of as <i>"(value of following policy $\pi$ at current state) = (value of next reward, which is determined from following $\pi$) + (discounted value at next state if we continue following $\pi$)"</i>.

We can also define the **action-value function**, or **Q-value** of state $s$ and action $a$ following policy $\pi$:

$$
Q_\pi(s,a) = \mathbb{E}\left[ \sum_{i=0}^\infty \gamma^i r_{t+i+1} \Bigg| s_t=s, a_t=a   \right]
$$

which can be written recursively much like the value function can:

$$
Q_\pi(s,a) = \sum_{s'} T(s' \mid s, a) \bigg( R(s, a, s') + \gamma \sum_{a'} \pi(a' \mid s') Q_\pi(s', a') \bigg)
$$

This equation can be thought of as <i>"(value of choosing particular action $a$ in current state $s$, then afterwards following $\pi$) = (value of next reward, which is determined from action $a$) + (discounted value at next state if we continue following $\pi$)"</i>. In other words it's conceptually the same equation as before but rewritten in terms of $Q$, conditioning on what the next action is before we go back to following $\pi$.

<details>
<summary>Question - what do you think is the formula relating V and Q to each other?</summary>

$V_\pi(s)$ is the value at a given state according to policy $\pi$, but this must be the average of all the values of taking action $a$ at this state, weighted by the probability that they're taken (under $\pi$). So we have:

$$
V_\pi(s) = \sum_a \pi(a \mid s) Q_\pi(s, a)
$$

Substituting this into the Bellman equation for the Q-value, we get:

$$
Q_\pi(s,a) = \sum_{s'} T(s' \mid s, a) \big[ R(s, a, s') + \gamma V_\pi(s') \big]
$$

</details>

### Optimal policies

* Two policies $\pi_1$ and $\pi_2$ are **equivalent** if $V_{\pi_1}(s) = V_{\pi_2}(s)$ for all states $s$. 
* A policy $\pi_1$ is **better** than, or **pareto dominates**, $\pi_2$ (denoted $\pi_1 \geq \pi_2$) if $V_{\pi_1}(s) \geq V_{\pi_2}(s)$ for all states $s$.

This gives a **partial order** on policies, as it may be the case that two policies are not equivalent, nor is one better than the other ($\pi_1$ might have higher value than $\pi_2$ in some states, and lower value in others).

A policy is **optimal** if it is better than all other policies. Optimal policies $\pi_1^*, \pi_2^*, \ldots$ are denoted with an asterisk to distinguish them from non-optimal policies. Note an optimal policy may not necessarily be unique, but the optimal value function $V^*(s) := \sup_\pi V_{\pi}(s)$ is unique by definition, and is achieved by at least one policy (exists $\pi^*$ such that $V^*= V_{\pi^*}$).

Theorem: An optimal deterministic policy exists for any MDP with $\gamma < 1$.

<details>
<summary>Proof (Optional) <i> TLDR - Define a metric space of bounded functions $V: \mathcal{S} \to \mathbb{R}$, and show 
that the Bellman operator is a contraction mapping on it. </i> 
</summary>

We prove that the optimal value function is unique, and show by construction the existence of an optimal policy.

Define the **Bellman optimality operator** $\mathcal{B}$ acting on a value function $V : \mathcal{S} \to \mathbb{R}$ as:

$$
(\mathcal{B} V)(s) = \max_{a \in \mathcal{A}} \sum_{s' \in \mathcal{S}} T(s' \mid s,a) \Big[ R(s,a,s') + \gamma V(s') \Big]
$$

- $\mathcal{B}$ maps bounded functions $\mathcal{S} \to \mathbb{R}$ to bounded functions $\mathcal{S} \to \mathbb{R}$.
- $\mathcal{B}$ is a **contraction mapping** in the sup-norm with the contraction factor 
equal to the discount factor $\gamma < 1$:

$$
\|\mathcal{B}V_1 - \mathcal{B}V_2\|_\infty \le \gamma \|V_1 - V_2\|_\infty
$$

for any two value functions $V_1$ and $V_2$, and where the sup-norm is defined as:

$$
\|V\|_\infty = \max_{s \in \mathcal{S}} |V(s)|
$$
and the difference between two value functions is defined in the obvious way:
$$
(V_1 - V_2)(s) = V_1(s) - V_2(s)
$$

Because $\mathcal{B}$ is a contraction on the complete metric space of bounded functions $V: \mathcal{S} \to \mathbb{R}$, it has a **unique fixed point** $V^*$ such that:

$$
\mathcal{B} V^* = V^*
$$

By the Bellman equation, $V_\pi(s)$ is an average over actions $a \sim \pi(\cdot \mid s)$ of $\sum_{s'} T(s' \mid s,a) \big[ R(s,a,s') + \gamma V_\pi(s') \big]$, and an average is at most the max over actions, so $V_\pi \leq \mathcal{B} V_\pi$.

$\mathcal{B}$ is also **monotone**: if $V_1(s) \leq V_2(s)$ for all $s$, then $(\mathcal{B}V_1)(s) \leq (\mathcal{B}V_2)(s)$ for all $s$. Applying $\mathcal{B}$ repeatedly therefore gives $V_\pi \leq \mathcal{B}V_\pi \leq \mathcal{B}^2 V_\pi \leq \ldots$, and since this sequence converges to the unique fixed point $V^*$, we get $V^*(s) \geq V_\pi(s)$ for all states $s$, and any policy $\pi$.


Define a **greedy policy** w.r.t. $V^*$ as:

$$
\pi^*(s) \in \arg\max_{a \in \mathcal{A}} \sum_{s' \in \mathcal{S}} T(s' \mid s,a) \Big[ R(s,a,s') + \gamma V^*(s') \Big]
$$

- This policy achieves $V^*$ for all states:

$$
V_{\pi^*}(s) = V^*(s), \quad \forall s \in \mathcal{S}
$$

This isn't quite by definition, but it only takes one line. Write $\mathcal{B}_{\pi^*}$ for the Bellman operator of the fixed policy $\pi^*$, i.e. $(\mathcal{B}_{\pi^*}V)(s) = \sum_{s'} T(s' \mid s,\pi^*(s)) \big[ R(s,\pi^*(s),s') + \gamma V(s') \big]$. Since $\pi^*$ picks a maximising action, $\mathcal{B}_{\pi^*} V^* = \mathcal{B} V^* = V^*$, so $V^*$ is a fixed point of $\mathcal{B}_{\pi^*}$. But $\mathcal{B}_{\pi^*}$ is also a $\gamma$-contraction (the same argument as for $\mathcal{B}$, without the max), so its fixed point is unique, and by the Bellman equation that fixed point is $V_{\pi^*}$. Hence $V_{\pi^*} = V^*$.

Note it may not be unique, as there may be multiple actions that achieve the maximum value.

</details>

### Exercise - reason about toy environment

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-25 minutes on this exercise.
> This will help build your intuition for these formulas and expressions.
> ```

Consider the following environment: There are two actions $A = \{a_L, a_R\}$, three states $S = \{s_0, s_L, s_R\}$ and three rewards $R = \{0,1,2\}$. The environment is deterministic, and can be represented by the following transition diagram:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/markov-diagram.png" width="400">

The edges represent the state transitions given an action, as well as the reward received. For example, in state $s_0$, taking action $a_L$ means the new state is $s_L$, and the reward received is $+1$. The transitions leading out from $s_L$ and $s_R$ are independent of action. This gives us effectively two choices of deterministic policies, $\pi_L(s_0) = a_L$ and $\pi_R(s_0) = a_R$ (it is irrelevant what those policies do in the other states.)

Your task here is to compute the value $V_{\pi}(s_0)$ for $\pi = \pi_L$ and $\pi = \pi_R$. Which policy is better? Does the answer depend on the choice of discount factor $\gamma$? If so, how?

<details>
<summary>Answer</summary>

Following the first policy, this gives
$$
V_{\pi_L}(s_0) = 1 + \gamma V_{\pi_L}(s_L) = 1 + \gamma(0 + \gamma V_{\pi_L}(s_0)) = 1 +\gamma^2 V_{\pi_L}(s_0)
$$
Rearranging, this gives
$$
V_{\pi_L}(s_0) = \frac{1}{1-\gamma^2}
$$
Following the second policy, this gives
$$
V_{\pi_R}(s_0) = 0 + \gamma V_{\pi_R}(s_R) = \gamma(2 + \gamma V_{\pi_R}(s_0)) = 2 \gamma + \gamma^2 V_{\pi_R}(s_0)
$$
Rearranging, this gives
$$
V_{\pi_R}(s_0) = \frac{2\gamma}{1-\gamma^2}
$$
Therefore,
$$
\pi^* = \begin{cases}
\pi_L & \gamma < 1/2 \\
\pi_L \text{ or } \pi_R & \gamma = 1/2 \\
\pi_R & \gamma > 1/2
\end{cases}
$$
which makes sense, an agent that discounts heavily ($\gamma < 1/2$) is shortsighted,
and will choose the reward 1 now, over the reward 2 later.
</details>

## Tabular RL, Known Environments

For the moment, we focus on environments for which the agent has access to $T$ and $R$, which will allow us to solve the Bellman equation explicitly. 

We will also assume that the policy $\pi$ is a deterministic mapping from state to action, and denote $a_t = \pi(s_t)$.
This means we can drop the sum $\sum_a$ in the Bellman equation, and write:

$$
V_\pi(s) = \sum_{s'} T(s' \mid s, \pi(s)) \Big( R(s, \pi(s), s') + \gamma V_\pi (s') \Big)
$$

and in $Q$-value form:

$$
\begin{aligned}
Q_\pi(s,a) &= \sum_{s'} T(s' \mid s,a) \left( R(s, a, s') + \gamma Q_\pi (s', \pi(s')) \right) \\ 
&= \sum_{s'} T(s' \mid s,a) \Big( R(s, a, s') + \gamma V_\pi(s') \Big)
\end{aligned}
$$

Below, we've provided a simple environment for a gridworld taken from [Russell and Norvig](http://aima.cs.berkeley.edu/). The agent can choose one of four actions: `up` (0), `right` (1), `down` (2) and `left` (3), encoded as numbers. The observation is just the state that the agent is in (states are labelled left-to-right, top-to-bottom, from 0 to 11).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/gridworld.png" width="300">

The result of choosing an action will move the agent in that direction, unless they would bump into a wall 
$\fcolorbox{grey}{grey}{\phantom{-1}}$ or walk outside the grid, in which case the agent doesn't move. Both the $\fcolorbox{lightgreen}{lightgreen}{\textcolor{black}{+1}}$ (heaven) and 
$\fcolorbox{red}{red}{\textcolor{black}{-1}}$ (hell) terminal states "trap" the agent, and any movement from one of the terminal states leaves the agent in the same state, and no reward is received. The agent receives a small penalty for each move $r = -0.04$ to encourage them to move to a terminal state quickly, unless the agent is moving into a terminal state, in which case they receive reward either $+1$ or $-1$ as appropriate.

Lastly, the environment is slippery, and the agent only has a 70% chance of moving in the direction chosen, with a 10% chance each of attempting to move in the other three cardinal directions.

<details>
<summary>From the bottom right cell, there are two possible routes to heaven: A short path that 
goes past hell, and the path that goes the long way around the wall. 
Which one do you think the agent will prefer? How do you think changing the penalty value r=-0.04 to something else might affect this?
</summary>

The long route requires more steps, which (assuming $r<0$) will be penalised more, but due to the 30% probability of slipping, walking near
the $\fcolorbox{red}{red}{\textcolor{black}{-1}}$ means the agent might fall into it by mistake, so the agent should avoid getting
anywhere close to it.

We can guess that there is a certain threshold value $r^*$ such that the optimal policy follows the short route for harsher penalty $r<r^*$ (as movement penalty is high, so it's worth risking the short route), and the long route for milder penalty $r>r^*$.

You can test this out (and find the approximate value of $r^*$) when you run your RL agent!
</details>

Provided is a class that allows us to define environments with known dynamics. 

The attributes you will need are:

* `.num_states : int` which gives the number of states
* `.num_actions : int` which gives the number of actions
* `.T : Float[np.ndarray, "num_states num_actions num_states"]` which represents the transitional probability $T(s_{\text{new}} \mid s, a)$ = `T[s, a, s_new]`
* `.R : Float[np.ndarray, "num_states num_actions num_states"]` which represents the reward function $R(s,a,s_{\text{new}})$ = `R[s,a,s_new]` associated with entering state $s_{\text{new}}$ from state $s$ by taking action $a$.

This environment also provides two additional parameters which we will not use now, but needed later where the environment is treated as a black box, and agents learn from interaction.

* `.start`, the state with which interaction with the environment begins. By default, assumed to be state zero.
* `.terminal`, a list of all the terminal states (with which interaction with the environment ends). By default, assumed that there are no terminal states.

**This class is already imported for you from `utils.py` - don't copy it into your notebook.** The version below is abridged for reading (for example it omits the `_build` method that constructs `T` and `R`), so pasting it would overwrite the full version and break the environments you define on top of it in this notebook, like `Toy` and `CliffWalking`.

<details>
<summary>Environment class (abridged, for reference only - already imported from utils.py)</summary>

```python
# no need to copy in, already provided in utils.py
class Environment:
    def __init__(self, num_states: int, num_actions: int, start=0, terminal=None):
        self.num_states = num_states
        self.num_actions = num_actions
        self.start = start
        self.terminal = np.array([], dtype=int) if terminal is None else terminal
        (self.T, self.R) = self._build()
        # self.T : (num_states, num_actions, num_states) transition probabilities
        # self.R : (num_states, num_actions, num_states) reward function
        
    def render(self, pi: Arr):
        """
        Takes a policy pi, and draws an image of the behavior of that policy, if applicable.

        Args:
            pi : (num_states,) a policy

        Returns:
            None
        """
        raise NotImplementedError()
        
    def dynamics(self, state: int, action: int) -> tuple[Arr, Arr, Arr]:
        """
        Computes the distribution over possible outcomes for a given state
        and action.

        Args:
            state  : int (index of state)
            action : int (index of action)

        Returns:
            states  : (m,) all the possible next states
            rewards : (m,) rewards for each next state transition
            probs   : (m,) likelihood of each state-reward pair
        """
        raise NotImplementedError()
```

</details>

For example, here is the toy environment from the theory exercises earlier (the **3-state environment**) implemented in this format.

*Note - don't spend too much time reading this code and the plots generated below; this is just designed to illustrate how we can implement a particular environment on top of our class above by defining certain environment dynamics. The environment you'll actually be working with is the gridworld environment described above.*

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/markov-diagram.png" width="400">

In [3]:
class Toy(Environment):
    def dynamics(self, state: int, action: int):
        """
        Sets up dynamics for the toy environment:
            - In state s_L, we move to s_0 & get +0 reward regardless of action
            - In state s_R, we move to s_0 & get +2 reward regardless of action
            - In state s_0,
                - action LEFT=0 leads to s_L & get +1,
                - action RIGHT=1 leads to s_R & get +0
        """
        (SL, S0, SR) = (0, 1, 2)
        LEFT = 0

        assert 0 <= state < self.num_states and 0 <= action < self.num_actions

        if state == S0:
            (next_state, reward) = (SL, 1) if action == LEFT else (SR, 0)
        elif state == SL:
            (next_state, reward) = (S0, 0)
        elif state == SR:
            (next_state, reward) = (S0, 2)
        else:
            raise ValueError(f"Invalid state: {state}")

        return (np.array([next_state]), np.array([reward]), np.array([1]))

    def __init__(self):
        # start in s_0 (state 1): it's the only state with a choice to make
        super().__init__(num_states=3, num_actions=2, start=1)

Given a definition for the `dynamics` function, the `Environment` class automatically generates `T` and `R` for us. We can plot these below:

- The first plot shows **transition probabilities** $T(s_{next} \mid s, a)$. It has only probabilities of 1 and 0 because transitions are deterministic ($s_{next}$ is a function of $s$ and $a$). At $s_L$ or $s_R$ you'll move back to the center $s_0$ regardless of what action you take, but at $s_0$ you can move either left or right and it will have different outcomes.
- The second plot shows **rewards** $R(s, a, s_{next})$. The rewards are $1$ for $s_0 \to s_L$, and $2$ for $s_R \to s_0$, and zero reward for all other possible transitions.

Note that the environment being deterministic means the reward is uniquely determined by $s$ and $a$ (because $s_{next}$ is a function of $s$ and $a$). This won't always be the case in RL, as we move to more complex environments.

In [4]:
toy = Toy()

actions = ["a_L", "a_R"]
states = ["s_L", "s_0", "s_R"]

imshow(
    toy.T,  # dimensions (s, a, s_next)
    title="Transition probabilities T(s_next | s, a) for toy environment",
    facet_col=0,
    facet_labels=[f"Current state is s = {s}" for s in states],
    y=actions,
    x=states,
    labels={
        "x": "Next state, s_next",
        "y": "Action taken, a",
        "color": "Transition<br>Probability",
    },
    text_auto=".0f",
    border=True,
    width=850,
    height=350,
)

imshow(
    toy.R,  # dimensions (s, a, s_next)
    title="Rewards R(s, a, s_next) for toy environment",
    facet_col=0,
    facet_labels=[f"Current state is s = {s}" for s in states],
    y=actions,
    x=states,
    labels={"x": "Next state, s_next", "y": "Action taken, a", "color": "Reward"},
    text_auto=".0f",
    border=True,
    width=850,
    height=350,
)

We provide a general `GridWorld` class that builds an environment - its states, transitions and rewards - directly from an **ASCII map**, so a new layout is just a string rather than a bespoke `dynamics` method. 

You'll only ever call its constructor and the two visualisation methods - here's that interface (the full source, including `dynamics`, is in `utils.py`):

**Again, don't copy this into your notebook** - `GridWorld` is already imported from `utils.py`, and this interface has no method bodies, so pasting it would break `Norvig` below (e.g. with an error that `num_states` isn't defined).

```python
class GridWorld(Environment):
    def __init__(self, grid_map: str, step_reward=0.0, goal_reward=1.0, trap_reward=-1.0,
                 cliff_reward=-100.0, slipperiness=0.0):
        """
        Build a gridworld from an ASCII map, e.g.

            GridWorld(
                "S..\n"
                ".#.\n"
                "..G"
            )

        Map characters: 'S' start, 'G' goal (+goal_reward, terminal), 'T' trap (+trap_reward,
        terminal), 'C' cliff (+cliff_reward, teleports back to the start, NOT terminal), '#' wall
        (impassable - moving into it leaves you in place), '.' empty floor.

        Args:
            step_reward:   reward for every non-terminal transition (0 -> sparse; < 0 -> step penalty)
            goal_reward:   reward for entering a 'G' cell
            trap_reward:   reward for entering a 'T' cell
            cliff_reward:  reward for stepping into a 'C' cell (you're also sent back to the start)
            slipperiness:  if > 0, the chosen action succeeds w.p. (1 - slipperiness); otherwise a
                           random other direction is taken (0.3 mimics Norvig; 0 is deterministic)
        """

    def render(self, pi: Arr):
        """Print the gridworld as rows of emoji tiles (terrain, or policy `pi`'s arrows on the floor)."""

    def show_map(self, pi: Arr | None = None, title: str = "Gridworld map", show: bool = True):
        """Plot the gridworld as a grid of emoji tiles: 🏁 start, 🟩 goal, 🟥 trap, ⬛ wall, 🟫 cliff,
        ⬜ floor. If a policy `pi` is passed, floor tiles instead show its action arrow (⬆️➡️⬇️⬅️)."""
```

Apart from the constructor, `GridWorld` inherits the `Environment` interface the planning code relies on - `.T`, `.R`, `.num_states`, `.num_actions`, `.start`, `.terminal` - all built from the ASCII map.

The **Norvig** gridworld below is just this class with a specific map and `slipperiness=0.3` (so the chosen action succeeds 70% of the time).

In [5]:
class Norvig(GridWorld):
    """The 3x4 Russell & Norvig gridworld, expressed as a `GridWorld` map: a +1 goal in the top-right
    with a -1 trap directly below it, a wall in the middle, a small step penalty everywhere else, and
    `slipperiness=0.3` (the chosen action succeeds 70% of the time)."""

    def __init__(self, penalty=-0.04):
        super().__init__(
            """
            ...G
            .#.T
            S...
            """,
            step_reward=penalty,
            goal_reward=1.0,
            trap_reward=-1.0,
            slipperiness=0.3,
        )


norvig = Norvig()
tests.test_norvig_gridworld_equivalence(Norvig)

# `render` prints the grid; `show_map` plots it. Here we show a random policy:
pi_random = np.random.randint(0, norvig.num_actions, (norvig.num_states,))
norvig.show_map(pi_random, title="Norvig gridworld: random policy")

All tests in `test_norvig_gridworld_equivalence` passed!


## Policy Evaluation

At the moment, we would like to determine the value function $V_\pi$ of some policy $\pi$. **We will assume policies are deterministic**, and encode policies as a lookup table from states to actions (so $\pi$ will be a vector of length `num_states`, and $\pi(i)$ is the action we take at state $i$).

Firstly, we will use the Bellman equation as an update rule: Given a current estimate $\hat{V}_\pi$ of the value function $V_{\pi}$, we can obtain a better estimate by using the Bellman equation, sweeping over all states.
$$
\forall s. \hat{V}_\pi(s) \leftarrow \sum_{s'} T(s' \,|\, s, a) \left( R(s,a,s') + \gamma \hat{V}_\pi(s') \right) \;\text{ where } a = \pi(s)
$$
Recall that the true value function $V_\pi$ satisfies this as an equality, i.e. it is a fixed point of the iteration. So we can start with $\hat{V}_\pi(s) = 0$ for all $s$ as our initial guess, and continue applying this update rule until the result stabilizes: $\max_s |\hat{V}^{new}(s) - \hat{V}^{old}(s)| < \epsilon$ for some small $\epsilon > 0$.

<details>
<summary>
Why does iterating the Bellman equation converge to the correct answer?
</summary>

We proved above in a drop-down that the Bellman optimality operator is a contraction mapping; the same argument (with the max over actions replaced by the fixed action $\pi(s)$) shows that this policy-evaluation update is also a contraction with factor $\gamma$, and so the iteration converges to the (unique) fixed point, and the fixed point is the exact value function, as we proved that it satisfies the Bellman equation.
</details>

### Exercise - implement `policy_eval_numerical`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> ```

The `policy_eval_numerical` function takes in a deterministic policy $\pi$ and transition & reward matrices $T$ and $R$, and computes the value function $V_\pi$ using the Bellman equation as an update rule. It keeps iterating, computing estimates $\hat{V}^1_\pi, \hat{V}^2_\pi, \ldots$ until it reaches `max_iterations`, or we terminate after $n$ iterations, where
$n$ is the first value for which $|| \hat{V}_\pi^{n+1} - \hat{V}_\pi^n ||_\infty < \epsilon$ - which ever comes first.

Recall - we store our transition and reward matrices as `env.T` and `env.R`, where we define `env.T[s, a, s']` $= T(s' \,|\, s, a)$ and `env.R[s, a, s']` $= R(s, a, s')$.

In [6]:
def policy_eval_numerical(env: Environment, pi: Arr, gamma=0.99, eps=1e-8, max_iterations=10_000) -> Arr:
    """
    Numerically evaluates the value of a given policy by iterating the Bellman equation
    Args:
        env: Environment
        pi : shape (num_states,) - The policy to evaluate
        gamma: float - Discount factor
        eps  : float - Tolerance
        max_iterations: int - Maximum number of iterations to run
    Outputs:
        value : float (num_states,) - The value function for policy pi
    """
    num_states = env.num_states

    value = np.zeros((num_states,))
    transition_matrix = env.T[range(num_states), pi, :]  # shape [s, s_next]
    reward_matrix = env.R[range(num_states), pi, :]  # shape [s, s_next]

    for i in range(max_iterations):
        new_value = einops.reduce(transition_matrix * (reward_matrix + gamma * value), "s s_next -> s", "sum")

        delta = np.abs(new_value - value).max()
        if delta < eps:
            break
        value = np.copy(new_value)
    else:
        print(f"Failed to converge after {max_iterations} steps.")

    return value


tests.test_policy_eval(policy_eval_numerical, exact=False)

All tests in `test_policy_eval` passed!


<details>
<summary>Help - I'm stuck on the iterative formula</summary>

You can start by indexing `env.T[range(num_states), pi, :]`, which effectively gives you the **transition matrix** with shape `(num_states, num_states)` with values $T(s' \,|\, s, \pi(s))$ for all $s, s'$ (since the action $\pi(s)$ is a deterministic function of the starting state $s$). You can do exactly the same for `env.R` to get the **reward matrix**.

If you prefer, you can also write this using Callum's `eindex` notation:
```
T_pi = eindex(env.T, pi, "s [s] s_new -> s s_new")
```

Next, you can apply the update formula by multiplying these tensors and summing over $s'$. Be careful of tensor shapes and broadcasting!

</details>


<details><summary>Solution</summary>

```python
def policy_eval_numerical(env: Environment, pi: Arr, gamma=0.99, eps=1e-8, max_iterations=10_000) -> Arr:
    """
    Numerically evaluates the value of a given policy by iterating the Bellman equation
    Args:
        env: Environment
        pi : shape (num_states,) - The policy to evaluate
        gamma: float - Discount factor
        eps  : float - Tolerance
        max_iterations: int - Maximum number of iterations to run
    Outputs:
        value : float (num_states,) - The value function for policy pi
    """
    num_states = env.num_states

    value = np.zeros((num_states,))
    transition_matrix = env.T[range(num_states), pi, :]  # shape [s, s_next]
    reward_matrix = env.R[range(num_states), pi, :]  # shape [s, s_next]

    for i in range(max_iterations):
        new_value = einops.reduce(transition_matrix * (reward_matrix + gamma * value), "s s_next -> s", "sum")

        delta = np.abs(new_value - value).max()
        if delta < eps:
            break
        value = np.copy(new_value)
    else:
        print(f"Failed to converge after {max_iterations} steps.")

    return value
```
</details>

## Exact Policy Evaluation

Essentially what we are doing in the previous step is solving the Bellman equation via iteration. However, we can actually solve it directly in this case, since it's effectively just a set of linear equations! In future cases we won't be able to because we won't know all the transition probabilities & rewards, but here we can solve directly.

To do this, let's define the **transition matrix** $\mathbf{P}^\pi$ and **reward matrix** $\mathbf{R}^\pi$ just like we did in the solution to the previous exercise:
$$
\mathbf{P}^\pi_{i,j} = T(j \,|\, i, \pi(i)) \quad \mathbf{R}^\pi_{i,j} = R(i, \pi(i), j)
$$
$\mathbf{P}^\pi$ represents the transition probabilities from one state to the next assuming we follow policy $\pi$, 
and $\mathbf{R}^\pi$ is the reward when we perform this transition. 
Let $\mathbf{v}^\pi$ be the vector representing the value function for policy $\pi$, i.e. $\mathbf{v}^\pi_s := V^\pi(s)$ then we can write our Bellman equation as:
$$
\mathbf{v}_s^\pi = \sum_{s'} \mathbf{P}^\pi_{s, s'} (\mathbf{R}^\pi_{s, s'} + \gamma \mathbf{v}_{s'}^\pi)
$$
Let's define $\mathbf{r}^\pi$ as $\mathbf{r}_s^\pi = \sum_{s'} \mathbf{P}^\pi_{s, s'} \mathbf{R}^\pi_{s, s'}$ - this is a weighted average of our next-state rewards according to their transition probabilities, in other words its the expected next-state reward for all possible starting states:
$$
\mathbf{r}^\pi_s = \sum_{s'} T(s' \mid s, \pi(s)) R(s, \pi(s), s')
$$
 We can then rewrite the above equation in matrix form as:
$$
\mathbf{v}^\pi = \mathbf{r}^\pi + \gamma \mathbf{P}^\pi \mathbf{v}^\pi
$$
and rearrange to solve for $\mathbf{v}^\pi$ directly:
$$
\mathbf{v}^\pi = (I - \gamma \mathbf{P}^\pi)^{-1} \mathbf{r}^\pi
$$
This inverse is guaranteed to exist whenever $\gamma < 1$ (see 
the dropdown below for intuition as to why).

<details>
<summary>Intuition for this equation</summary>

One nice intuition comes from the Taylor expansion $(\mathbf{I} - \mathbf{X})^{-1} = \mathbf{I} + \mathbf{X} + \mathbf{X}^2 + \ldots$ which is valid for any matrix $\mathbf{X}$ with $||\mathbf{X}|| < 1$. Applying this to our equation, we get:
$$
\mathbf{v}^\pi=\left(\mathbf{I}+\gamma \mathbf{P}^\pi+\gamma^2 \left(\mathbf{P}^\pi\right)^2+\gamma^3 \left(\mathbf{P}^\pi\right)^3+\ldots\right) \mathbf{r}^\pi
$$
which is just an infinite sum of the accumulated reward, discounted by $\gamma$ at each step (since if $P$ is a transition matrix for 1 step then $P^2$ describes the transition matrix for 2 steps, and so on).

</details>

### Exercise - implement `policy_eval_exact`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

The function below should apply the exact policy evaluation formula to solve for $\mathbf{v}^\pi$ directly.

You can start by defining the same `transition_matrix` and `reward_matrix` as in the previous exercise. You may also find the function `np.linalg.solve` helpful: given matrix equation $\mathbf{A} \mathbf{x} = \mathbf{b}$, then `np.linalg.solve(A,b)` returns `x`, assuming a solution exists.

In [7]:
def policy_eval_exact(env: Environment, pi: Arr, gamma=0.99) -> Arr:
    """
    Finds the exact solution to the Bellman equation.
    """
    num_states = env.num_states

    transition_matrix = env.T[range(num_states), pi, :]  # shape [s, s_next]
    reward_matrix = env.R[range(num_states), pi, :]  # shape [s, s_next]

    r = einops.einsum(transition_matrix, reward_matrix, "s s_next, s s_next -> s")

    mat = np.eye(num_states) - gamma * transition_matrix

    return np.linalg.solve(mat, r)


tests.test_policy_eval(policy_eval_exact, exact=True)

All tests in `test_policy_eval` passed!


<details><summary>Solution</summary>

```python
def policy_eval_exact(env: Environment, pi: Arr, gamma=0.99) -> Arr:
    """
    Finds the exact solution to the Bellman equation.
    """
    num_states = env.num_states

    transition_matrix = env.T[range(num_states), pi, :]  # shape [s, s_next]
    reward_matrix = env.R[range(num_states), pi, :]  # shape [s, s_next]

    r = einops.einsum(transition_matrix, reward_matrix, "s s_next, s s_next -> s")

    mat = np.eye(num_states) - gamma * transition_matrix

    return np.linalg.solve(mat, r)
```
</details>

## Policy Improvement

Now we can evaluate a policy, we want to improve it. One way we can do this is to compare how well $\pi$ performs on a state versus the value obtained by choosing another action instead, that is, the Q-value. If there is an action $a'$ for which $Q_{\pi}(s,a') > V_\pi(s) \equiv Q_\pi(s, \pi(s))$, then action $a'$ is preferable over action $\pi(s)$ in state $s$. 
Recall that we could write out our Bellman equation like this:

$$
Q_\pi(s,a) = \sum_{s'} T(s' \mid s,a) \left( R(s, a, s') + \gamma V_\pi(s') \right)
$$

which gives us an update rule for our policy $\pi$, given that we've evaluated $V_\pi$ using our closed-form expression from earlier, by choosing
a better action according to what maximises the RHS of the Bellman equation.
Or in other words:

$$
\pi^\text{better}(s) = \text{argmax}_a \sum_{s'} T(s' \mid s, a) (R(s,a,s') + \gamma V_{\pi}(s'))
$$

### Exercise - implement `policy_improvement`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-25 minutes on this exercise.
> ```

The `policy_improvement` function takes in a value function $V$, and transition and reward tensors $T$ and $R$, and returns a new policy $\pi^\text{better}$.

Recall, we've defined our objects as `env.T[s, a, s']` $= T(s' \,|\, s, a)$, and `env.R[s, a, s']` $= R(s, a, s')$.

Your returned policy should be an array of `dtype=int` (one action per state), rather than floats - it represents **the deterministic action you take in each state**.

In [8]:
def policy_improvement(env: Environment, V: Arr, gamma=0.99) -> Arr:
    """
    Args:
        env: Environment
        V  : (num_states,) value of each state following some policy pi
    Outputs:
        pi_better : vector (num_states,) of actions representing a new policy obtained via policy
                    iteration
    """
    q_values_for_every_state_action_pair = einops.einsum(env.T, env.R + gamma * V, "s a s_next, s a s_next -> s a")
    pi_better = q_values_for_every_state_action_pair.argmax(axis=1)
    return pi_better


tests.test_policy_improvement(policy_improvement)

All tests in `test_policy_improvement` passed!


<details>
<summary>Solution</summary>

```python
def policy_improvement(env: Environment, V: Arr, gamma=0.99) -> Arr:
    """
    Args:
        env: Environment
        V  : (num_states,) value of each state following some policy pi
    Outputs:
        pi_better : vector (num_states,) of actions representing a new policy obtained via policy
                    iteration
    """
    q_values_for_every_state_action_pair = einops.einsum(env.T, env.R + gamma * V, "s a s_next, s a s_next -> s a")
    pi_better = q_values_for_every_state_action_pair.argmax(axis=1)
    return pi_better
```

Note, the reason our solution works is that computing `env.R + gamma * V` correctly broadcasts `V`: it has dimensions `(s_next,)`, and `env.R` has dimensions `(s, a, s_next)`.

</details>

Putting these together, we now have an algorithm to find the optimal policy for an environment.

$$
\pi_0 \overset{E}{\to} V_{\pi{_0}}
\overset{I}{\to} \pi_1 \overset{E}{\to} V_{\pi_{1}}
\overset{I}{\to} \pi_2 \overset{E}{\to} V_{\pi_{2}} \overset{I}{\to}  \ldots
$$

We alternate policy evaluation ($\overset{E}{\to}$) and policy improvement ($\overset{I}{\to}$), with each improvement step being a **strict** monotonic improvement,
$$
\pi_0 < \pi_1 < \pi_2 < \ldots
$$
until the policy no longer changes ($\pi_n = \pi_{n+1}$), at which point we have a fixed point of the optimal Bellman equations, 
and thus an optimal policy:
$$
V_{\pi^*}(s) = \max_a \sum_{s'} T(s' \mid s, a) (R(s,a,s') + \gamma V_{\pi^*}(s'))
$$

### Exercise - implement `find_optimal_policy`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> ```

Implement the `find_optimal_policy` function below. This should iteratively do the following:

* find the correct value function $V_{\pi}$ for your given policy $\pi$ using your `policy_eval_exact` function,
* update your policy using `policy_improvement`,

The first policy $\pi_0$ is arbitrary (we've picked all zeros).

A few things to note:

* Don't forget that policies should be of `dtype=int`, rather than floats! $\pi$ in this case represents **the deterministic action you take in each state**.
* Since optimal policies may not be unique, the automated tests will merely check that your optimal policy has the same value function as the optimal policy found by the solution.

In [9]:
def find_optimal_policy(env: Environment, gamma=0.99, max_iterations=10_000):
    """
    Args:
        env: environment
    Outputs:
        pi : (num_states,) int, of actions representing an optimal policy
    """
    pi = np.zeros(shape=env.num_states, dtype=int)

    for i in range(max_iterations):
        V = policy_eval_exact(env, pi, gamma)
        pi_new = policy_improvement(env, V, gamma)
        if np.array_equal(pi_new, pi):
            return pi_new
        else:
            pi = pi_new
    else:
        print(f"Failed to converge after {max_iterations} steps.")
        return pi


tests.test_find_optimal_policy(find_optimal_policy)

penalty = -0.04
norvig = Norvig(penalty)
pi_opt = find_optimal_policy(norvig, gamma=0.99)
norvig.show_map(pi_opt, title="Optimal Policy, penalty = -0.04")

All tests in `test_find_optimal_policy` passed!


<details><summary>Solution</summary>

```python
def find_optimal_policy(env: Environment, gamma=0.99, max_iterations=10_000):
    """
    Args:
        env: environment
    Outputs:
        pi : (num_states,) int, of actions representing an optimal policy
    """
    pi = np.zeros(shape=env.num_states, dtype=int)

    for i in range(max_iterations):
        V = policy_eval_exact(env, pi, gamma)
        pi_new = policy_improvement(env, V, gamma)
        if np.array_equal(pi_new, pi):
            return pi_new
        else:
            pi = pi_new
    else:
        print(f"Failed to converge after {max_iterations} steps.")
        return pi
```
</details>

Once you've passed the tests, you should play around with the `penalty` value for the gridworld environment and see how this affects the optimal policy found. Which squares change their optimal strategy when the penalty takes different negative values (e.g. -0.04, -0.1, or even -1). To see why this happens, refer back to the dropdown earlier, in the section "Tabular RL, Known Environments". Can you see why the different rewards lead to the behaviours that they do?

## Bonus

- Implement and test your policy evaluation method on other environments.
- Complete some exercises in Chapters 3 and 4 of Sutton and Barto.
- Modify the tabular RL solvers to allow stochastic policies or to allow $\gamma=1$ on episodic environments (may need to change how environments are defined.)

# 2️⃣ Learning

> ##### Learning Objectives
>
> * Understand the ideas behind temporal difference methods such as SARSA and Q-learning.
> * Implement SARSA and Q-Learning, and compare them on different environments.
> * Understand the TD($\lambda$) algorithm, and how it can we used to mix over short and long timescale updates.

Now, we deal with situations where the environment is a black-box, and the agent must learn the rules of the world via interaction with it. This is different from everything else we've done so far, e.g. in the previous section we could calculate optimal policies by using the tensors $R$ and $T$, which we will now assume the agent doesn't have direct knowledge of.

We call algorithms which have access to the transition probability distribution and reward function **planning algorithms**, as they don't need to learn the dynamics of the environment but can directly plan what to do (like a chess-playing algorithm that does tree search). **Q-learning** is a **model-free algorithm**. It isn't given the environment's dynamics and doesn't try to learn them; it learns only which actions are good in which states. From the original paper introducing Q-learning:

> <i>[Q-learning] provides agents with the capability of learning to act optimally in Markovian domains by experiencing the consequences of actions, without requiring them to build maps of the domains.</i>

The "Q" part of Q-learning refers to the function $Q_\pi$ which we encountered earlier - the expected discounted sum of rewards for an action $a$ taken in a particular state $s$, based on some policy $\pi$.

For a lecture on the material in this section (temporal difference learning and Q-learning), watch the video below up to but not
including the "Beyond Tabular Learning" section.

<iframe width="540" height="304" src="https://www.youtube.com/embed/_4AHUdpkarM" frameborder="0" referrerpolicy="strict-origin-when-cross-origin" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

## Tabular Methods

The methods used here are called tabular methods, because they create a lookup table from `(state, action)` to the Q value. This is pure memorization, and if our reward function was sampled from the space of all functions, this is usually the best you can do because there's no structure that you can exploit to do better.

We can hope to do better on most "natural" reward functions that do have structure. For example in a game of poker, there is structure in both of the actions (betting £100 will have a similar reward to betting £99 or £101), and between states (having a pair of threes in your hand is similar to having a pair of twos or fours). We need to take advantage of this, otherwise there are just too many states and actions to have any hope of training an agent.

One idea is to use domain knowledge to hand-code a function that "buckets" states or actions into a smaller number of equivalence classes and use those as the states and actions in a smaller version of the problem (see [Sutton and Barto](https://www.andrew.cmu.edu/course/10-703/textbook/BartoSutton.pdf), Section 9.5). This was one component in the RL agent [Libratus: The Superhuman AI for No-Limit Poker](https://www.science.org/doi/epdf/10.1126/science.aao1733). The details are beyond the scope of today's material, but I found them fascinating.

If you don't have domain knowledge to leverage, or you care specifically about making your algorithm "general", we will see how to deal with this later: we offload the table to a neural network, and it will learn for us a suitable state aggregation method: generalizing to never-before-seen states based on ones it has already seen to work out how to act. The neural network is then trained on the experience of interacting with the environment.

## Optional Readings

The material provided together with the lecture cover everything you'll need, but here are some optional readings if you want to go deeper.

- [Sutton and Barto](https://www.andrew.cmu.edu/course/10-703/textbook/BartoSutton.pdf)
   - Chapter 6, Section 6.1, 6.3 (Especially Example 6.4)
   - Note that Section 6.1 talks about temporal difference (TD) updates for the value function $V$. We will instead be using TD updates for the Q-value $Q$.
   - Don't worry about the references to Monte Carlo in Chapter 5.
- [Q-Learning](https://link.springer.com/content/pdf/10.1007/BF00992698.pdf) The original paper where Q-learning is first described.

Again, we'll be using NumPy for this section, and we'll start off with our gridworld environment from the previous section:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/gridworld.png" width="300">

but this time we'll use it within the `gym` framework.

## Intro to OpenAI Gymnasium

Today, we'll be using `gymnasium` (version `1.x`), an actively maintained fork of OpenAI's no longer maintained `gym` library. This provides a uniform interface to many different RL environments including Atari games. The original `gym` was released in 2016 and details of the API have changed significantly over the years. If you're looking for source code, make sure you look at the [release](https://github.com/Farama-Foundation/Gymnasium/releases) matching your installed version (check `gym.__version__`), since older tutorials and Stack Overflow answers often describe a different API.

<!-- Gymnasium was for many years the standard library for RL, and many other libraries use the same interface,but more recently there's been others like [Brax](https://github.com/google/brax) and [PufferLib](https://github.com/pufferai/pufferlib) that allow -->

<details>
<summary>Note - <code>gymnasium</code> vs <code>gym</code></summary>

`gymnasium` is the new standard for RL environments, and we'll be using it for the rest of this chapter. The rest of the material will just discuss `gymnasium` directly, but we've included here a list of the important differences between `gym==0.23.1` (the version previously used in the ARENA exercises) and modern `gymnasium`, just in case you want to transfer your old code over (full details can be found [here](https://gymnasium.farama.org/introduction/migration_guide/)).

Gymnasium also made breaking changes between versions `0.29` and `1.0`, mostly to how *vectorized* environments reset and report terminal observations and episode statistics (see below), and to how the Atari environments are registered (you now need `import ale_py` before calling `gym.make("ALE/...")`). An earlier version of this course pinned `gymnasium==0.29.0` to avoid the vectorized-environment changes. That's no longer necessary: from day 2.2 onwards we use our own batched PyTorch environments (`gpu_env.py`, plus EnvPool for Atari and Brax for MuJoCo) rather than gymnasium's vector environments, so we can use gymnasium 1.x, which (unlike `0.29`) also installs on Python 3.12.

**5 objects returned in `env.step`, not 4**. In version `0.23.1` of the original gym library, the `env.step` method returned 4 objects: `observation, reward, done, info`. Implicitly, `done` was actually defined as `terminated or truncated`, where `truncated` is true if the episode has timed out and `terminated` is true if the episode ends for non-time related reasons (e.g. the agent has fallen off a cliff). In `gymnasium` (or more recent versions of `gym`), we split this into 2 variables, i.e. `env.step` returns the 5-tuple of `observation, reward, terminated, truncated, info`.

**2 objects returned in `env.reset`, not 1**. In version `0.23.1` of the original gym library, the `env.reset` method returned a single object: `obs` (the observation). In `gymnasium`, it returns 2 objects: `observation` and `info` (the latter of which is the same as the `info` object returned by the `step` method conventionally).

**Render mode**. You need to create video-based environments by passing `render_mode="rgb_array"` into `gym.make`. When you do this, you don't also need to call `mode="rgb_array"` in the `render` method.

**Vectorized environments** (only relevant if you use gymnasium's `gym.vector.SyncVectorEnv` etc. yourself - the environments we use from day 2.2 onwards don't). The `infos` object is now a dict of arrays (one entry per env), rather than a list of dicts. By default, gymnasium 1.x vector environments reset a finished sub-environment on the *next* call to `step`: the `next_obs` returned on the final step is the true terminal observation, and the following `step` ignores that sub-environment's action and returns the first observation of its new episode, so you need to be careful not to store that step as a real transition. With the `RecordEpisodeStatistics` wrapper, finished episodes' stats are in `infos["episode"]["r"]` and `infos["episode"]["l"]`, masked by `infos["_episode"]`. (In gymnasium `0.29`, vector environments reset immediately instead, putting the true terminal observation in `infos["final_observation"][env_idx]` and the episode stats in `infos["final_info"][env_idx]["episode"]`. Our own batched environments in `gpu_env.py` use this older, simpler layout.)

**Random seeding.** The `Env.seed()` method no longer exists, and is now replaced with `Env.reset(seed=seed)`.

<!-- *If you're happy continuing to work with `gym`, then skip this section!*

We're using a relatively old version of `gym` (0.23.1), rather than the more recent (and maintained `gymnasium`). This is because RL libraries are notoriously buggy and susceptible to breaking changes, and we're working with a version which we know is compatible with the material. Annoyingly this version of gym is only compatible with `wandb==0.13.10` rather than the more recent `0.16` versions, which is why we've fixed that version in the requirements file.

If you want to transition to `gymnasium`, it is possible (most of the material has been tested with gymnasium as well), but we chose not to make the transition because it wasn't worth all the breaking changes. A few bits of guidance if you want to use `gymanasium`:

- We recommend pinning version `gymnasium==0.29.0`.
- You should replace `import gym.(x)` with `import gymnasiumn.(x)`, and replace `import gym` with `import gymnasium as gym`.
- The most important change is that `env.step` no longer returns 4 objects, it returns 5: `done` is split into `terminated` and `truncated`.
	- `terminated` tells you whether the episode has ended because of environment-specific reasons (e.g. the cartpole has fallen over), and `truncated` tells you whether the episode has ended because the maximum number of steps has been reached. The relationship is `done = terminated or truncated`.
	- You can deal with this by simply defining `done = terminated or truncated` everywhere (or the `|` operator if you're dealing with arrays), and use `done` in the same way as before. However, you could also try and use `terminated` rather than `truncated` when appropriate. Using `done` will only incentivize the agent to maximize its reward within the episode length (e.g. it could always fall over at the 499th step in CartPole-v0 and this would still be fine), but using `terminated` will have different behavioural effects. Can you see why this is?
	- In day 1 material, you'll need to return a 5-tuple rather than a 4-tuple. Often this will just involve inserting `truncated=False` into the return tuple.
- `obs.reset()` now returns a 2-tuple of `obs, info` rather than just `obs`.
- The `infos` object is structured differently: it's a dict of lists, not a list of dicts.
	- You should replace `infos[environment]["episode"]` with `infos["final_info"][environment]["episode"]`, and `infos[environment]["terminal_observation"]` with `infos["final_observation"][environment]`.
- Remove `self.seed()` in the probes.
- Pass `render_mode="rgb_array"` into `gym.make` in the `make_envs` utils function.
- Comment out the `test_agent` function.
- When using atari, you'll want to use the gymnsasium library's built-in wrappers (found [here](https://gymnasium.farama.org/api/wrappers/misc_wrappers/)), not the stuff in `atari_wrappers.py`. -->

</details>

We're using NumPy for this section. PyTorch provides GPU support and autograd, neither of which is of any use to environment code or the code we're writing today.


#### The `step` method

The environment's `step` method takes the action selected by the agent and returns five values: `obs`, `reward`, `terminated`, `truncated`, and the `info` dictionary.

`obs` and `reward` is the next observation and reward that the agent receives based on the action chosen.

`terminated` indicates if the environment has entered a terminal state and ended. Here, both the goal-states (+1 and -1) are terminal. Early termination is equivalent to an infinite trajectory where the agent remains trapped for all future states, and always receives reward zero. `truncated` indicates the episode was instead cut short by the time limit; we often combine the two as `done = terminated or truncated`.

`info` can contain anything extra that doesn't fit into the uniform interface - it's up to the environment what to put into it. A good use of this is for debugging information that the agent isn't "supposed" to see, like the dynamics of the environment. Agents that cheat and peek at `info` are helpful because we know that they should obtain the maximum possible rewards; if they aren't, then there's a bug. We will throw the entire underlying environment into `info`, from which an agent could cheat by peeking at the values for `T` and `R`.


<details>
<summary>Terminology note - observation vs. state</summary>

We use the word *observation* here as some environments are *partially observable*, the agent receives not an exact description of the state they are in, but merely an observation giving a partial description (for our gridworld, it could be a description of which cells directly adjacent to the agent are free to move into, rather than precisely which state they are in). This means that the agent would be unable to distinguish the cell north of the wall from the cell south of the wall. Returning the state as the observation is a special case, and we will often refer to one or the other as required.
</details>

#### The `render` method

Render is only used for debugging or entertainment, and what it does is up to the environment. It might render a little popup window showing the Atari game, or it might give you a RGB image tensor, or just some ASCII text describing what's happening.

For our gridworld environment, it will print a small representation of the current policy by showing what action it chooses for each state.

#### Observation and Action Types

A `gym.Env` is a generic type: both the type of the observations and the type of the actions depends on the specifics of the environment. Right now, we're only dealing with the simplest case: a discrete set of actions which are the same in every state. In general, the actions could be continuous, or depend on the state. We're also just dealing with a single discrete observation here, although later we'll be taking arrays as our observations.

Below, we define a class that allows us to use our old environment definition from before, and wrap it in a `gym.Env` instance so we can learn from experience instead. Read the code below carefully and make sure you understand how the Gym environment API works.

In [10]:
ObsType: TypeAlias = int | np.ndarray
ActType: TypeAlias = int


class DiscreteEnviroGym(gym.Env):
    """
    A discrete environment class for reinforcement learning, compatible with OpenAI Gym.

    This class represents a discrete environment where actions and observations are discrete.
    It is designed to interface with a provided `Environment` object which defines the
    underlying dynamics, states, and actions.

    Attributes:
        action_space (gym.spaces.Discrete): The space of possible actions.
        observation_space (gym.spaces.Discrete): The space of possible observations (states).
        env (Environment): The underlying environment with its own dynamics and properties.
    """

    action_space: gym.spaces.Discrete
    observation_space: gym.spaces.Discrete

    def __init__(self, env: Environment):
        super().__init__()
        self.env = env
        self.observation_space = gym.spaces.Discrete(env.num_states)
        self.action_space = gym.spaces.Discrete(env.num_actions)
        self.reset()

    def step(self, action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        """
        Execute an action and return the new state, reward, terminated flag, truncated flag, and additional info.
        The behaviour of this function depends primarily on the dynamics of the underlying
        environment.
        """
        states, rewards, probs = self.env.dynamics(self.pos, action)
        idx = self.np_random.choice(len(states), p=probs)
        new_state, reward = states[idx], rewards[idx]
        self.pos = new_state
        terminated = self.pos in self.env.terminal
        truncated = False
        info = {"env": self.env}
        return new_state, reward, terminated, truncated, info

    def reset(self, seed: int | None = None, options=None) -> tuple[ObsType, dict]:
        """
        Resets the environment to its initial state.
        """
        super().reset(seed=seed)
        self.pos = self.env.start
        return self.pos, {}

    def render(self, mode="human"):
        assert mode == "human", f"Mode {mode} not supported!"

For gym, we also need to register an environment to use it, and we can also wrap environments so that each episode of interaction is limited to a fixed number of steps.

<details>
<summary>Registering an Environment</summary>

Usually we won't use a class like `DiscreteEnviroGym` directly, because we'll want to wrap it in classes that add various other features that help it work with the rest of the gymnasium library. The `register` function does this for us: it stores information about our `Env` in a registry so that a later call to `gym.make` can look it up using the `id` string that is passed in, and then create an instance of our class with a bunch of extra wrapper classes.

By convention, the `id` strings have a suffix with a version number. There can be multiple versions of the "same" environment with different parameters, and benchmarks should always report the version number for a fair comparison. For instance, `id="NorvigGrid-v0"` below.

</details>

<details>
<summary>TimeLimit Wrapper</summary>

One important wrapper class is the `TimeLimit` wrapper - this is used to ensure that the environment terminates after a fixed number of steps. It works by having the `truncated` flag returned by the `step` method be overridden after `max_episode_steps`, and set to `True`. You can see from looking at the `step` method above that without this, the agent might keep playing forever, because it always returns `terminated=truncated=False`. The time limit is also an essential part of the problem definition: if it were larger or shorter, there would be more or less time to explore, which means that different algorithms (or at least different hyperparameters) would then have improved performance.

</details>

In [11]:
gym.envs.registration.register(
    id="NorvigGrid-v0",
    entry_point=DiscreteEnviroGym,
    max_episode_steps=100,
    nondeterministic=True,
    kwargs={"env": Norvig(penalty=-0.04)},
)

gym.envs.registration.register(
    id="ToyGym-v0",
    entry_point=DiscreteEnviroGym,
    max_episode_steps=2,  # s_0 -> (s_L or s_R) -> s_0
    nondeterministic=False,
    kwargs={"env": Toy()},
)

## Agent class

Now, you'll start defining agents to act within this class.

We define a dataclass `Experience`, to store **experiences**, defined as $e_t = (s_t, a_t, r_{t+1}, s_{t+1}, a_{t+1})$. In the literature, this is often called a **percept**.

You'll be defining your various agents as subclasses of the base class `Agent`. It contains the following methods/attributes:

* `AgentConfig`, which allows us to more easily package up the arguments we use to initialize our agent.
* `get_action`, which returns the agent's next action, given the current observation.
    * This isn't implemented for `Agent`, because we'll implement it separately for each agent we build on top of the base agent.
* `observe`, which determines how the model updates the policy based on receiving experience.
    * The default behaviour is to do nothing (i.e never learn from experience).
* `reset`, which is run before each episode starts.
    * All that the base method does is reset the random number generator, but again future agents might need to do more, e.g. reset data gathered from the `observe` method.
* `run_episode`, which runs a single episode of interaction between an agent and the environment.
* `train`, which runs `run_episode` multiple times in series, and collects the sum of rewards for each, so we can observe improvements in agent performance.

We've also given you the `RandomAgent` subclass, which picks an action at random using the random number generator provided by gym. This is useful as a baseline to ensure the environment has no bugs. If your later agents are doing worse than random, you have a bug!

In [12]:
@dataclass
class Experience:
    """
    A class for storing one piece of experience during an episode run.
    """

    obs: ObsType
    act: ActType
    reward: float
    new_obs: ObsType
    new_act: ActType | None = None


@dataclass
class AgentConfig:
    """Hyperparameters for agents"""

    epsilon: float = 0.1
    lr: float = 0.05
    optimism: float = 0


defaultConfig = AgentConfig()


class Agent:
    """
    Base class for agents interacting with an environment.

    You do not need to add any implementation here.
    """

    rng: np.random.Generator

    def __init__(
        self,
        env: DiscreteEnviroGym,
        config: AgentConfig = defaultConfig,
        gamma: float = 0.99,
        seed: int = 0,
    ):
        self.env = env
        self.reset(seed)
        self.config = config
        self.gamma = gamma
        self.num_actions = env.action_space.n
        self.num_states = env.observation_space.n
        self.name = type(self).__name__

    def get_action(self, obs: ObsType) -> ActType:
        raise NotImplementedError()

    def observe(self, exp: Experience) -> None:
        """
        Agent observes experience, and updates model as appropriate.
        Implementation depends on type of agent.
        """
        pass

    def reset(self, seed: int) -> None:
        # The env is seeded with the same `seed`, and gymnasium's `env.np_random` is exactly `np.random.default_rng(seed)`.
        # Seeding our generator the same way would make the agent's random draws the very same numbers as the env's, so
        # we use a separate child stream.
        self.rng = np.random.default_rng(np.random.SeedSequence(seed).spawn(1)[0])

    def run_episode(self, seed) -> list[float]:
        """
        Simulates one episode of interaction, agent learns as appropriate

        Inputs:
            seed : Seed for the random number generator

        Returns:
            The rewards obtained during the episode
        """
        rewards = []
        obs, info = self.env.reset(seed=seed)
        self.reset(seed=seed)
        done = False
        while not done:
            act = self.get_action(obs)
            new_obs, reward, terminated, truncated, info = self.env.step(act)
            done = terminated or truncated
            exp = Experience(obs, act, reward, new_obs)
            self.observe(exp)
            rewards.append(reward)
            obs = new_obs
        return rewards

    def train(self, n_runs=500):
        """
        Run a batch of episodes, and return the total reward obtained per episode

        Inputs:
            n_runs : The number of episodes to simulate

        Returns:
            The discounted sum of rewards obtained for each episode
        """
        all_returns = []
        for seed in range(n_runs):
            rewards = self.run_episode(seed)
            all_returns.append(utils.sum_rewards(rewards, self.gamma))
        return all_returns


class Random(Agent):
    def get_action(self, obs: ObsType) -> ActType:
        return self.rng.integers(0, self.num_actions)

## Cheater Agent

First, you'll implement a cheating agent that peeks at the info and finds the optimal policy directly using your previous code. If your agent gets more than this in the long run, you have a bug!

### Exercise - implement `Cheater`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Below, you should implement the cheater agent. Recall the `find_optimal_policy` function to find the optimal policy for any environment and value of gamma. You should use this function to define the optimal policy on initialization, and then fill in the `get_action` method so that it always returns the corresponding optimal action.

Once you've done this, run the code below to check that your cheating agent outperforms the random agent. The cheating agent represents the best possible behavior, as it omnisciently always knows to play optimally without having to learn.

On the environment `ToyGym-v0`, (assuming $\gamma = 0.99$ and that the environment terminates after 2 timesteps) the cheating agent should always get reward $2 \gamma = 1.98$,
and the random agent should get a fluctuating reward, with average $\frac{2 \gamma + 1}{2} = 1.49$.

Hint: Use `env.unwrapped.env` to extract the `Environment` wrapped inside `gym.Env`, from which you can access `.T` and `.R`.

<details>
<summary>Help - I get 'AttributeError: 'DiscreteEnviroGym' object has no attribute 'num_states''.</summary>

This is probably because you're passing the `DiscreteEnviroGym` object to your `find_optimal_policy` function. In the following line of code:

```python
env_toy = gym.make("ToyGym-v0")
```

the object `env_toy` wraps around the `Toy` environment you used in the previous section. As mentioned, you'll need to use `env.unwrapped.env` to access this environment, and its dynamics.

</details>

In [13]:
class Cheater(Agent):
    def __init__(self, env: DiscreteEnviroGym, config: AgentConfig = defaultConfig, gamma=0.99, seed=0):
        super().__init__(env, config, gamma, seed)
        self.pi_opt = find_optimal_policy(self.env.unwrapped.env, self.gamma)

    def get_action(self, obs):
        return self.pi_opt[obs]


tests.test_cheater(Cheater)

env_toy = gym.make("ToyGym-v0")
agents_toy: list[Agent] = [Cheater(env_toy), Random(env_toy)]
returns_dict = {}
for agent in agents_toy:
    returns = agent.train(n_runs=100)
    returns_dict[agent.name] = utils.cummean(returns)

line(
    list(returns_dict.values()),
    names=list(returns_dict.keys()),
    title=f"Avg. reward on {env_toy.spec.name}",
    labels={"x": "Episode", "y": "Avg. reward", "variable": "Agent"},
    template="simple_white",
    width=700,
    height=400,
)

All tests in `test_cheater` passed!


<details><summary>Solution</summary>

```python
class Cheater(Agent):
    def __init__(self, env: DiscreteEnviroGym, config: AgentConfig = defaultConfig, gamma=0.99, seed=0):
        super().__init__(env, config, gamma, seed)
        self.pi_opt = find_optimal_policy(self.env.unwrapped.env, self.gamma)

    def get_action(self, obs):
        return self.pi_opt[obs]


env_toy = gym.make("ToyGym-v0")
agents_toy: list[Agent] = [Cheater(env_toy), Random(env_toy)]
returns_dict = {}
for agent in agents_toy:
    returns = agent.train(n_runs=100)
    returns_dict[agent.name] = utils.cummean(returns)

line(
    list(returns_dict.values()),
    names=list(returns_dict.keys()),
    title=f"Avg. reward on {env_toy.spec.name}",
    labels={"x": "Episode", "y": "Avg. reward", "variable": "Agent"},
    template="simple_white",
    width=700,
    height=400,
)
```
</details>

## SARSA: On-Policy TD Control

Now we wish to train an agent on the same gridworld environment as before, but this time it doesn't have access to the underlying dynamics (`T` and `R`). The rough idea here is to try and estimate the Q-value function directly from samples received from the environment. Recall that the optimal Q-value function satisfies

$$
\begin{aligned}
Q^*(s,a) &= \mathbb{E}_{\pi^*} \left[ G_t  \mid s_t = s, a_t = a\right] \\
&= \mathbb{E}_{\pi^*} \left[r + \gamma \max_{a'} Q^*(s', a')  \mid s_t = s, a_t = a\right] \\
\end{aligned}
$$

where

* $s'$ represents the next state after $s$,
* $a'$ the next action after $a$
* $r$ is the reward obtained from taking action $a$ in state $s$
* the expectation $\mathbb{E}_{\pi^*}$ is with respect to both the optimal policy $\pi^*$, as well as the stochasticity in the environment itself.

Intuitively, this last line of the formula says <i>"if our policy is optimal, then (current value if next action is $a$) = (reward from action $a$) + (discounted value of choosing the best possible action at the next step)"</i>.

This means that, for any particular episode $s_0, a_0, r_1, s_1, a_1, r_2, s_2, a_2, r_3,\ldots$ generated via interaction with an
optimal policy $\pi^*$, we have **in expectation** that:

$$
Q^*(s_t, a_t) \approx r_{t+1} + \gamma  Q^*(s_{t+1}, a'_{t+1})
$$

where $a'_{t+1} = \pi^*(s_{t+1}) = \text{argmax}_a Q^*(s_{t+1}, a)$. Since this approximation is only valid on the assumption that 
future decisions are made optimally, it will not hold for non-optimal policies, as it may not be the case that $a'_{t+1} = a_{t+1}$.

Letting $\widehat{Q}^*$ denote our best current estimate of $Q^*$, the error $\delta_t := r_{t+1} + \gamma \widehat{Q}^*(s_{t+1}, a_{t+1}) - \widehat{Q}^*(s_t,a_t)$  in this "guess" is called the **TD error**, and tells us in which direction we should bump our estimate of $Q^*$.
Of course, this estimate might be wildly inaccurate (even for the same state-action pair!), due to the stochasticity of the environment, or poor estimates of $Q^*$. So, we update our estimate slightly in the direction of $\delta_t$, much like stochastic gradient descent does. The update rule (with learning rate $\alpha > 0$) is

$$
\widehat{Q}^*(s_t,a_t) \leftarrow \widehat{Q}^*(s_t,a_t) + \alpha \left( r_{t+1} + \gamma \widehat{Q}^*(s_{t+1}, a_{t+1}) - \widehat{Q}^*(s_t,a_t) \right)
$$

This update depends on the information $(s_t, a_t, r_{t+1}, s_{t+1}, a_{t+1})$, and so is called **SARSA** learning (recall these are the exact 5 values we store in our `Experience` dataclass). Note that SARSA learns **on-policy**, in that it only learns from data that was actually generated by the current policy $\pi$, derived from the current estimate of $Q$, $\pi(s) = \text{argmax}_a \widehat{Q}^*(s,a)$.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/sarsa.png" width="700">

Note - we can intuitively describe this update step as "changing your $Q$-function based on the action you *actually* took, in a way which makes it more consistent with the current policy". On its own this wouldn't get us to the optimal value $Q^*$, instead it'd just get us to the correct value of $Q^\pi$. This is why we have to combine it with some kind of **policy improvement**. We use an epsilon-greedy policy (see below).

## Q-Learning: Off-Policy TD Control

As mentioned at the end of the last section, what SARSA is essentially doing is estimating $Q_\pi$ by using the rewards gathered by following policy $\pi$. But we don't actually care about $Q_\pi$, what we care about is $Q^*$. $Q$-Learning provides a slight modification to SARSA, by modifying the TD-error $\delta_t$ to use the action that $\pi$ *should* have taken in the next state $s_{t+1}$ (namely $\text{argmax}_a \widehat{Q}^*(s_{t+1},a)$) rather than the action $a_{t+1}$ that was actually taken there.

$$
\widehat{Q}^*(s_t,a_t) \leftarrow \widehat{Q}^*(s_t,a_t) + \alpha \left( r_{t+1} + \gamma \max_a \widehat{Q}^*(s_{t+1}, a) - \widehat{Q}^*(s_t,a_t) \right)
$$

Intuitively, if the SARSA update is "changing your $Q$-function based on the action you *actually* took, in a way which makes it more consistent with the current policy", then this update is "changing your $Q$-function based on the best possible action you could have taken, in a way which makes it more consistent with the optimal policy".

Note that each Q-learning update depends on the information $(s_t, a_t, r_{t+1}, s_{t+1})$. This means that $Q$-learning tries to estimate $Q^*$ directly, regardless of what policy $\pi$ generated the episode, and so Q-Learning learns *off-policy*.

Q-Learning in fact can eventually learn the optimal Q-value $Q^*$ for **any** policy $\pi$ such that every state-action pair gets visited infinitely often (plus some other conditions regarding the learning rate). This means Q-Learning will still work, even if the policy is completely random! (Learning will be **very** slow, however.)


<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/qlearn.png" width="700">

## Explore vs. Exploit

Lastly, methods to learn the Q-value when acting with a greedy policy often have trouble exploring. If a state-action pair $(s,a)$ with low Q-value $Q^*(s,a)$ just so happens to return a high reward by chance, the greedy policy with respect to $Q$ will often choose action $a$ in state $s$ instead of exploring potentially other good actions first. To remedy this, we use instead an $\epsilon$-greedy policy with respect to the current Q-value estimates: With probability $\epsilon$, a random action is chosen, and with probability $1-\epsilon$ the greedy action $\text{argmax}_a Q(s,a)$ is chosen. The exploration probability $\epsilon$ is a hyperparameter that for now we will set to a constant $\epsilon = 0.1$, but more sophisticated techniques include the use of a schedule to start exploring often early, and then decay exploration as time goes on.

We also have the choice of how the estimate $\widehat{Q}^*(s,a)$ is initialized. By choosing "optimistic" values (initial values that are much higher than what we expect $Q^*(s,a)$ to actually be), this will encourage the greedy policy to hop between different actions in each state when they discover they weren't as valuable as first thought.

### Exercise - Implement Q-learning and SARSA

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 20-30 minutes on these exercises.
> 
> They are difficult, so you should use the hints if you need.
> ```

You should fill in the classes `EpsilonGreedy`, `QLearning` and `SARSA` below.

- `EpsilonGreedy`
    - This is simply a base class which keeps track of the current Q-value estimates, and selects an action based on an epsilon-greedy policy (i.e. with probability $\epsilon$ a random action is taken, and with probability $1-\epsilon$ the Q-maximizing action is taken, according to current Q-value estimates).
    - We've already filled in the initialization for you (with optimism), all that's left is for you to implement epsilon-greedy action selection.
- `QLearning`
    - You should fill in the `observe` method for the `QLearning` class, to update Q-values based on observations (according to the formula above).
- `SARSA`
    - You should fill in the `observe` method for the `SARSA` class, to update Q-values based on observations (according to the formula above).

Note that we've given the SARSA agent a slightly different `run_episode` function. Can you see what's different about this, and why we've done this? (This is pretty subtle so don't worry about it too much.)

<details>
<summary>Answer - what's different?</summary>

We've reordered the `while` loop content. Before, it looked like this:

```python
while not done:
    act = self.get_action(obs)
    new_obs, reward, terminated, truncated, info = self.env.step(act)
    done = terminated or truncated
    exp = Experience(obs, act, reward, new_obs)
    self.observe(exp)
    rewards.append(reward)
    obs = new_obs
```

Now, it looks like this:

```python
while not done:
    new_obs, reward, terminated, truncated, info = self.env.step(act)
    done = terminated or truncated
    new_act = self.get_action(new_obs)
    exp = Experience(obs, act, reward, new_obs, new_act)
    self.observe(exp)
    rewards.append(reward)
    obs = new_obs
    act = new_act
```

</details>

<details>
<summary>Answer - why is it different?</summary>

It's different in SARSA because the order of operations is different. We have:

- **SARSA** = take action $a_t$, observe $(r_{t+1}, s_{t+1})$, then decide on next action $a_{t+1}$, then update
- **Q-learning** = take action $a_t$, observe $(r_{t+1}, s_{t+1})$, then update, then decide on next action $a_{t+1}$

The key point being that *the action $a_{t+1}$ has to be computed before the SARSA update step, since SARSA updates on actions taken*. Whereas Q-learning updates should happen after the action $a_t$.

Don't worry if you don't fully understand this, the order of operations for these 2 algorithms can get a bit messy. All you really need to understand is this: in both cases, the code variables `exp.obs, exp.act, exp.reward, exp.new_obs` in the observe methods correspond to the pseudocode variables $S, A, R, S'$, and in the case of SARSA we also have `exp.new_act` which corresponds to $A'$.

</details>

#### Tips

- Use `self.rng.random()` to generate random numbers in the range $[0,1)$, and `self.rng.integers(0, n)` for random integers in the range $0, 1, \ldots, n-1$.
- The random agent results in very long episodes, which slows evaluation. You can remove them from the experiment once you've convinced yourself that your agents are doing something reasonable.
- Leave $\gamma =0.99$ for now.


When you run the code below, you should expect SARSA and Q-Learning to perform similarly on this grid. Averaged over 200 seeds, Q-Learning is slightly ahead during the first ~200 episodes (by about 0.05) and the two end up level, but that gap is small next to the seed-to-seed noise, so with only 5 seeds either one may come out ahead. Both should be well below the cheater at first and close most of the gap as they learn, and both should be *much* better than the random agent.

In [14]:
class EpsilonGreedy(Agent):
    """
    A class for SARSA and Q-Learning to inherit from.
    """

    def __init__(
        self,
        env: DiscreteEnviroGym,
        config: AgentConfig = defaultConfig,
        gamma: float = 0.99,
        seed: int = 0,
    ):
        super().__init__(env, config, gamma, seed)
        self.Q = np.zeros((self.num_states, self.num_actions)) + self.config.optimism
        # Terminal states have value 0. We never act from them so their Q-values are never updated, but
        # Q-learning & SARSA bootstrap from them, so they mustn't keep the optimistic initial value.
        if hasattr(env.unwrapped, "env"):  # our `DiscreteEnviroGym` envs expose their terminal states
            self.Q[env.unwrapped.env.terminal] = 0

    def get_action(self, obs: ObsType) -> ActType:
        """
        Selects an action using epsilon-greedy with respect to Q-value estimates
        """
        if self.rng.random() < self.config.epsilon:
            return self.rng.integers(0, self.num_actions)
        else:
            return self.Q[obs].argmax()


class QLearning(EpsilonGreedy):
    def observe(self, exp: Experience) -> None:
        s_t, a_t, r_t_1, s_t_1 = exp.obs, exp.act, exp.reward, exp.new_obs
        self.Q[s_t, a_t] += self.config.lr * (r_t_1 + self.gamma * np.max(self.Q[s_t_1]) - self.Q[s_t, a_t])


class SARSA(EpsilonGreedy):
    def observe(self, exp: Experience):
        s_t, a_t, r_t_1, s_t_1, a_t_1 = exp.obs, exp.act, exp.reward, exp.new_obs, exp.new_act
        self.Q[s_t, a_t] += self.config.lr * (r_t_1 + self.gamma * self.Q[s_t_1, a_t_1] - self.Q[s_t, a_t])

    def run_episode(self, seed) -> list[float]:
        rewards = []
        obs, info = self.env.reset(seed=seed)
        self.reset(seed=seed)
        act = self.get_action(obs)
        done = False
        while not done:
            new_obs, reward, terminated, truncated, info = self.env.step(act)
            done = terminated or truncated
            new_act = self.get_action(new_obs)
            exp = Experience(obs, act, reward, new_obs, new_act)
            self.observe(exp)
            rewards.append(reward)
            obs = new_obs
            act = new_act
        return rewards


tests.test_epsilon_greedy(EpsilonGreedy)
tests.test_q_learning(QLearning)
tests.test_sarsa(SARSA)

gamma = 0.99
n_runs = 400
n_seeds = 5
norvig_specs = [
    (Cheater, AgentConfig(), "Cheater"),
    (QLearning, AgentConfig(), "Q-Learning"),
    (SARSA, AgentConfig(), "SARSA"),
    (Random, AgentConfig(), "Random"),
]
# average over a few seeds (tabular RL is noisy) and plot mean ± standard error
curves = utils.seeded_curves(norvig_specs, "NorvigGrid-v0", gamma=gamma, n_runs=n_runs, n_seeds=n_seeds)
title = f"Avg. reward on NorvigGrid-v0 (mean ± se, {n_seeds} seeds)"
utils.plot_learning_curves(curves, title=title, yaxis_title="Avg. reward", band="se")

All tests in `test_epsilon_greedy` passed!
All tests in `test_q_learning` passed!
All tests in `test_sarsa` passed!


<details><summary>Solution</summary>

```python
class EpsilonGreedy(Agent):
    """
    A class for SARSA and Q-Learning to inherit from.
    """

    def __init__(
        self,
        env: DiscreteEnviroGym,
        config: AgentConfig = defaultConfig,
        gamma: float = 0.99,
        seed: int = 0,
    ):
        super().__init__(env, config, gamma, seed)
        self.Q = np.zeros((self.num_states, self.num_actions)) + self.config.optimism
        # Terminal states have value 0. We never act from them so their Q-values are never updated, but
        # Q-learning & SARSA bootstrap from them, so they mustn't keep the optimistic initial value.
        if hasattr(env.unwrapped, "env"):  # our `DiscreteEnviroGym` envs expose their terminal states
            self.Q[env.unwrapped.env.terminal] = 0

    def get_action(self, obs: ObsType) -> ActType:
        """
        Selects an action using epsilon-greedy with respect to Q-value estimates
        """
        if self.rng.random() < self.config.epsilon:
            return self.rng.integers(0, self.num_actions)
        else:
            return self.Q[obs].argmax()


class QLearning(EpsilonGreedy):
    def observe(self, exp: Experience) -> None:
        s_t, a_t, r_t_1, s_t_1 = exp.obs, exp.act, exp.reward, exp.new_obs
        self.Q[s_t, a_t] += self.config.lr * (r_t_1 + self.gamma * np.max(self.Q[s_t_1]) - self.Q[s_t, a_t])


class SARSA(EpsilonGreedy):
    def observe(self, exp: Experience):
        s_t, a_t, r_t_1, s_t_1, a_t_1 = exp.obs, exp.act, exp.reward, exp.new_obs, exp.new_act
        self.Q[s_t, a_t] += self.config.lr * (r_t_1 + self.gamma * self.Q[s_t_1, a_t_1] - self.Q[s_t, a_t])

    def run_episode(self, seed) -> list[float]:
        rewards = []
        obs, info = self.env.reset(seed=seed)
        self.reset(seed=seed)
        act = self.get_action(obs)
        done = False
        while not done:
            new_obs, reward, terminated, truncated, info = self.env.step(act)
            done = terminated or truncated
            new_act = self.get_action(new_obs)
            exp = Experience(obs, act, reward, new_obs, new_act)
            self.observe(exp)
            rewards.append(reward)
            obs = new_obs
            act = new_act
        return rewards
```
</details>

Compare the performance of SARSA and Q-Learning on the gridworld environment v.s. the cheating agent and the random agent. Try to tune the hyperparameters to get the best performance you can.

- Which seems to work better? SARSA or Q-Learning?
- Does the optimism parameter seem to help?
- What's the best choice of exploration parameter $\epsilon$?
- The feedback from the environment is very noisy. At the moment, the code provided plots the cumulative average reward per episode. You might want to try plotting a sliding average instead, or an exponential weighted moving average (see `ewma` in `part3_ppo/utils.py`).

## When does exploration actually matter?

The Norvig gridworld above doesn't motivate $\epsilon$-greedy exploration well, because the
environment is so noisy we basically get random moves *for free*. A purely greedy agent ($\epsilon = 0$) gets knocked
around enough that it explores sufficiently well.

To see *why* we actually need $\epsilon > 0$, we want a **deterministic** environment, where the only way the agent ever visits a new state is by choosing to. Conveniently, the `GridWorld` maps we built the Norvig grid from are deterministic unless we pass `slipperiness > 0` - so we just need a new map with no slip.

Now consider this deterministic map. The start `S` is in the bottom-left, the goal `G` (reward `+1`) sits near the
top-right, and there's a trap `T` (reward `-1`) in the top-right corner; every other step gives reward `0`.

In [15]:
explore_grid = GridWorld("....T\n"
                        "...G.\n"
                        ".....\n"
                        ".....\n"
                        "S....")
explore_grid.show_map(show=True, title="Deterministic Exploration World")

gym.envs.registration.register(
    id="ExplorationGrid-v0",
    entry_point=DiscreteEnviroGym,
    max_episode_steps=100,
    nondeterministic=False,
    kwargs={"env": explore_grid},
)

Here's the catch. We initialise $Q$ to zeros, and `np.argmax` breaks ties by returning the *first* maximal entry - which is action `0`, "up". So a strictly greedy agent ($\epsilon = 0$) starts by walking straight up into the top wall, where "up" keeps it in place. Every action looks equally good (all $Q$-values are still `0`), so it just keeps choosing "up" forever, never receives any reward, and never learns a thing. The only way to ever reach the goal is to *deliberately try a different action* - which is exactly what $\epsilon > 0$ provides.

Run the cell below to sweep a range of exploration rates $\epsilon \in \{0, 0.1, 0.2, 0.5\}$ on this environment. Tabular RL is noisy, so we average each learning curve over 5 seeds and shade a $\pm 1$ standard-error envelope around it.

In [16]:
gamma = 0.99
n_runs = 100
n_seeds = 5
epsilons = [0.0, 0.1, 0.2, 0.5]

# one Q-learning agent per epsilon, averaged over several seeds (tabular RL is noisy), plotted mean ± se
explore_specs = [(QLearning, AgentConfig(epsilon=eps, lr=0.1, optimism=0.0), f"epsilon={eps}") for eps in epsilons]
curves = utils.seeded_curves(explore_specs, "ExplorationGrid-v0", gamma=gamma, n_runs=n_runs, n_seeds=n_seeds)
title = f"Q-learning on ExplorationGrid-v0 (mean ± se, {n_seeds} seeds)"
utils.plot_learning_curves(curves, title=title, band="se")

The $\epsilon = 0$ agent flatlines at zero return, as it never tries to explore. Agents with any $\epsilon > 0$ should
*eventually* learn to play optimally, though the speed at which they learn can vary with the exploration rate.

These envelopes also show that the *amount* of exploration matters. With $\epsilon = 0$ every seed follows the same deterministic path, so the envelope has zero width and the curve never leaves zero return. Once $\epsilon > 0$ the agents start reaching the goal and the curves climb. More exploration finds the goal sooner, so at first $\epsilon = 0.5$ is ahead. But there's a trade-off: with $\epsilon = 0.5$ the agent keeps taking random, suboptimal actions even after it has learned the way, so its per-episode return levels off below that of the $\epsilon = 0.1$ and $0.2$ runs. On this cumulative-average plot that shows up as the $\epsilon = 0.5$ curve flattening while the others keep climbing; within 100 episodes they only catch up with it.

## Eligibility Traces

Both SARSA and Q-Learning work well for environments that never terminate, as they update on each time step. Suppose instead we were focused on an episodic environment. 
Given a full trajectory from such an environment
$$
\tau = s_0, a_0, r_1, s_1, a_1, r_2, s_2, a_2, \ldots, s_{T-1}, a_{T-1}, r_T, s_T
$$
where $s_T$ is the terminal state (so no action is taken there). Recalling that the Q-value was defined as the expected discounted sum of rewards,
$$
Q(s, a) = \mathbb{E}_{\pi} \left[ G_t \mid s_t = s, a_t = a \right]
$$
and that the return $G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots + \gamma^{T-t-1} r_{T}$ is the sum of the rewards from now till the end of the episode, we 
defined an update rule for SARSA: 
$$
\begin{align*}
Q(s_t, a_t) &\leftarrow Q(s_t, a_t) + \alpha \delta_t \\
\delta_t &= r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t) \equiv \delta_{t:t+1}
\end{align*}
$$
under the assumption that the Q-value of the next state-action pair is a reasonable estimate of the rest of the return.
$$
Q(s_{t+1}, a_{t+1}) \approx G_{t+1} = r_{t+2} + \gamma r_{t+3} + \cdots + \gamma^{T-t-2} r_{T}
$$
Recalling that $G_t = r_{t+1} + \gamma r_{t+2} + \ldots$ is the standard return, 
let us define the partial estimated return as
$$
G_{t:t+n} = r_{t+1} + \gamma r_{t+2} + \cdots + \gamma^{n-1} r_{t+n} + \gamma^n Q(s_{t+n}, a_{t+n})
$$
so the standard TD error can be expressed as 
$$
\delta_{t} = G_{t:t+1} - Q(s_t, a_t)
$$
This is called a 1-step TD update. One could imagine instead using a 2-step TD update
$$
\begin{align*}
\delta_{t:t+2} &= r_{t+1} + \textcolor{red}{\gamma r_{t+2} + \gamma^2 Q(s_{t+2}, a_{t+2})} - Q(s_t, a_t) \\
&= G_{t:t+2} - Q(s_t, a_t)
\end{align*}
$$
which now depends on $(s_t, a_t, r_{t+1}, r_{t+2}, s_{t+2}, a_{t+2})$, or a full $n$-step TD update
$$
\delta_{t:T} = G_{t:T} - Q(s_t, a_t) = G_t - Q(s_t, a_t)
$$
which depends on the rest of the trajectory until the end of the episode.
Note that the partial return $G_{t:T}$ will not contain a term of the form $\max_a \gamma^{T-t} Q(s_T, a)$ as $s_T$ is terminal,
and there is no action to take, so the Q-value of the terminal state is 0.

What to choose? Shorter updates $\delta_{t:t+1}$ have less variance as they rely on fewer samples, but they might have a systematic bias as we replace
the tail of the return with our current best guess $\gamma Q(s_{t+1}, a_{t+1})$. Longer updates $\delta_{t:T}$ are unbiased as they use the actual true return, but have much higher variance due to the sum of many (potentially) noisy rewards.

One can try for a mixture to get the best of both worlds: We could use 
$$
\frac{1}{2} \delta_{t:t+1} + \frac{1}{2} \delta_{t:T} = \frac{1}{2} G_{t:t+1} + \frac{1}{2} G_{t:T} - Q(s_t, a_t)
$$
and this would still be a valid update rule, or we could use any convex mixture
$$
\theta_{t+1} G_{t:t+1} + \theta_{t+2} G_{t:t+2} + \ldots + \theta_{T} G_{t:T} - Q(s_t, a_t)
$$
for parameters so long as $\sum_{k=1}^{T-t} \theta_{t+k} = 1$ and $\theta_{t+k} \geq 0$.

The trick now is that we choose the weights $\theta_{t+k} = (1-\lambda) \lambda^{k-1}$ for $k=1,2,\ldots,T-t$,
and define the $\lambda$-return as
$$
G^\lambda_t = (1-\lambda) \sum_{k=1}^{\infty} \lambda^{k-1} G_{t:t+k}
$$
for continuing environments, or 
$$
G^\lambda_t = (1-\lambda) \sum_{k=1}^{T-t} \lambda^{k-1} G_{t:t+k} + \lambda^{T-t} G_t
$$
for terminating environments, where $\lambda$ is a hyperparameter that controls how hard we decay the importance of longer partial returns.
Then the update rule becomes
$$
Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left( G^\lambda_t - Q(s_t, a_t) \right)
$$
This is called the TD($\lambda$) algorithm. For $\lambda=0$, the update rule is the same as the 1-step TD update, and for $\lambda=1$, the update rule is the same
as using the full return $\delta_{t:T}$.

The problem with this update rule is that 

1. computing $G^\lambda_t$ naively is very expensive, requiring $O(T^2)$ time,
2. we would need to wait till the end of the episode to learn anything. For continuing environments with no end, we could never update!

Thankfully, there is a fix! We can use something called an **eligibility trace** to compute this update in an online fashion.
The eligibility trace $e_t(s,a)$ measures how we should "blame" a state-action pair on timestep $t$ for the error in the update.
When we visit a state-action pair, its eligibility trace is incremented, and the eligibility trace for all states decays over time geometrically.
We can define the eligibility trace as
$$
\begin{align*}
e_0(s,a) &:= 0 \\
e_t(s,a) &:= \gamma \lambda e_{t-1}(s,a) + \mathbf{1}(s = s_t, a = a_t)
\end{align*}
$$
for all states $s$ and actions $a$.

<details>
<summary>Derivation of the eligibility trace (very optional)</summary>

**NOTE:** This is an LLM-generated proof, checked by Claude Opus 4.8, but not checked by humans.

### Full Derivation of SARSA($\lambda$) Eligibility Trace Update

We want to show that the **forward-view $\lambda$-return update**:

$$
\Delta Q(s_t, a_t) = \alpha \left( G^\lambda_t - Q(s_t, a_t) \right)
$$

is equivalent to the **backward-view update** using eligibility traces:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \, \delta_t \, e_t(s,a),
$$

where

$$
e_0(s,a) = 0, \qquad e_t(s,a) = \gamma \lambda e_{t-1}(s,a) + \mathbf{1}\{(s,a) = (s_t, a_t)\},
$$

and 

$$
\delta_t = r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)
$$

is the standard 1-step TD error.

---

#### 1. The $\lambda$-return and partial n-step returns

For an episode of length $T$, the $\lambda$-return is defined as a weighted combination of n-step returns:

$$
G^\lambda_t = (1-\lambda) \sum_{n=1}^{T-t} \lambda^{\,n-1} G_{t:t+n} + \lambda^{T-t} G_t,
$$

where

$$
G_{t:t+n} = r_{t+1} + \gamma r_{t+2} + \cdots + \gamma^{n-1} r_{t+n} + \gamma^n Q(s_{t+n}, a_{t+n}).
$$

The forward-view update at time $t$ is

$$
\Delta Q(s_t, a_t) = \alpha \bigl(G^\lambda_t - Q(s_t, a_t)\bigr).
$$

---

#### 2. Express the n-step returns in terms of 1-step TD errors

Define the 1-step TD error:

$$
\delta_k = r_{k+1} + \gamma Q(s_{k+1}, a_{k+1}) - Q(s_k, a_k).
$$

Then the n-step return can be written as a sum of TD errors:

$$
G_{t:t+n} - Q(s_t, a_t) = \sum_{k=0}^{n-1} \gamma^k \delta_{t+k}.
$$

Substituting into the $\lambda$-return:

$$
\begin{aligned}
G^\lambda_t - Q(s_t, a_t) 
&= (1-\lambda) \sum_{n=1}^{T-t} \lambda^{\,n-1} \sum_{k=0}^{n-1} \gamma^k \delta_{t+k} + \lambda^{T-t} \sum_{k=0}^{T-t-1} \gamma^k \delta_{t+k} \\
&= \sum_{k=0}^{T-t-1} (\gamma \lambda)^k \delta_{t+k}.
\end{aligned}
$$

This identity shows that the $\lambda$-return minus the current Q-value equals a **geometrically-weighted sum of future 1-step TD errors**.

---

#### 3. Sum of forward-view updates over an episode

Consider the total change to a particular $Q(s,a)$ over an episode:

$$
\Delta Q_{\text{forward}}(s,a) = \sum_{t: (s_t,a_t)=(s,a)} \alpha \left( G^\lambda_t - Q(s_t,a_t) \right)
= \alpha \sum_{t: (s_t,a_t)=(s,a)} \sum_{k=t}^{T-1} (\gamma\lambda)^{k-t} \delta_k.
$$

Swap the order of summation (sum over TD times $k$ first):

$$
\Delta Q_{\text{forward}}(s,a) = \alpha \sum_{k=0}^{T-1} \delta_k \sum_{t=0}^{k} \mathbf{1}\{(s_t,a_t)=(s,a)\} (\gamma\lambda)^{k-t}.
$$

---

#### 4. Recognize the eligibility trace

The inner sum is exactly the **accumulating eligibility trace**:

$$
e_k(s,a) = \sum_{t=0}^{k} \mathbf{1}\{(s_t,a_t)=(s,a)\} (\gamma\lambda)^{k-t},
$$

which is equivalent to the recursive definition:

$$
e_k(s,a) = \gamma\lambda e_{k-1}(s,a) + \mathbf{1}\{(s_k,a_k)=(s,a)\}.
$$

Thus, the total update can be written as:

$$
\Delta Q_{\text{forward}}(s,a) = \alpha \sum_{k=0}^{T-1} \delta_k \, e_k(s,a),
$$

which is exactly the sum of the backward-view incremental updates.

Note that steps 2-4 treat $Q$ as fixed for the whole episode, i.e. they assume **offline** updates, where the increments $\alpha \delta_k e_k(s,a)$ are accumulated and only applied to $Q$ at the end of the episode. Under that assumption the two views are exactly equivalent. The algorithm below (and our implementation) applies each increment **online**, straight away, so the later TD errors $\delta_k$ are computed with slightly updated Q-values, and the backward view only approximates the forward view (the approximation is good for small $\alpha$). Getting exact equivalence for online updates needs a modified trace, as in "true online TD($\lambda$)" ([van Seijen & Sutton, 2014](https://proceedings.mlr.press/v32/seijen14.html); Sutton & Barto, section 12.5).

---

#### 5. Practical backward-view SARSA($\lambda$) algorithm

At each time step $t$:

1. Observe $(s_t, a_t, r_{t+1}, s_{t+1}, a_{t+1})$.
2. Compute the TD error:

$$
\delta_t = r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)
$$

3. Update eligibility traces for all $(s,a)$:

$$
e_t(s,a) = \gamma \lambda e_{t-1}(s,a) + \mathbf{1}\{(s,a)=(s_t,a_t)\}
$$

4. Update Q-values:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \, \delta_t \, e_t(s,a)
$$

This yields the **online, incremental backward-view SARSA($\lambda$) update**: exactly equivalent to the forward-view $\lambda$-return update for offline updates, and a close approximation to it for the online updates we use.

</details>

This then gives the update rule for SARSA($\lambda$) on timestep $t$ as
$$
\begin{align*}
\delta_t &= r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t) \\
e(s_t, a_t) &\leftarrow e(s_t, a_t) + 1 \\
\forall s. \forall a. Q(s, a) &\leftarrow Q(s, a) + \alpha \delta_t e_t(s, a) \\
\forall s. \forall a. e(s, a) &\leftarrow \gamma \lambda e(s, a)
\end{align*}
$$
simply the original 1-step update rule, but with the TD-errors decayed according to the eligibility trace.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/img/sarsa_lambda.png" width="800">

Note that in the algorithm, the inner-most loop over all state-action pairs
can be parallelized, as the updates are independent. In practice, we store the eligibility trace $e$ 
as a lookup table `e[s,a]`, and broadcast the updates.

### Exercise - Implement SARSA($\lambda$)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-20 minutes on these exercises.
> 
> This shouldn't be too hard as it's mostly mild changes to the SARSA algorithm before.
> ```

You should fill in the class `SARSA_lambda` below. 

* We can reuse the same `run_episode` function as the `SARSA` class, but modifying to first reset 
the eligibility trace `e` back to zero. 
* The policy will be epsilon-greedy as before, so we can also reuse the `get_action` method.
* The `observe` function will need to be modified for the new SARSA($\lambda$) update rule.
* The `init` method should initialize the eligibility trace to zero, as well as the Q-values (either directly or via `super().__init__`).

In [17]:
@dataclass
class TD_LambdaConfig(AgentConfig):
    lambda_: float = 0.95


defaultTDLambdaConfig = TD_LambdaConfig()


class SARSA_lambda(SARSA):
    def __init__(
        self,
        env: DiscreteEnviroGym,
        config: TD_LambdaConfig = defaultTDLambdaConfig,
        gamma: float = 0.99,
        seed: int = 0,
    ):
        super().__init__(env, config, gamma, seed)
        self.lambda_ = config.lambda_
        self.e = np.zeros((self.num_states, self.num_actions), dtype=np.float32)

    def run_episode(self, seed) -> list[float]:
        self.e[:, :] = 0
        return super().run_episode(seed)

    def observe(self, exp: Experience):
        s_t, a_t, r_t, s_tp1, a_tp1 = exp.obs, exp.act, exp.reward, exp.new_obs, exp.new_act
        delta_t = r_t + self.gamma * self.Q[s_tp1, a_tp1] - self.Q[s_t, a_t]
        self.e[s_t, a_t] += 1

        self.Q += self.config.lr * delta_t * self.e  # broadcast update
        self.e *= self.gamma * self.lambda_


tests.test_sarsa_lambda(SARSA_lambda)

All tests in `test_sarsa_lambda` passed!


In [18]:
gamma = 0.99
n_runs = 400
n_seeds = 5
norvig_lambda_specs = [
    (Cheater, AgentConfig(), "Cheater"),
    (QLearning, AgentConfig(), "Q-Learning"),
    (SARSA, AgentConfig(), "SARSA"),
    (SARSA_lambda, TD_LambdaConfig(), "SARSA(λ)"),
    (Random, AgentConfig(), "Random"),
]
# average over a few seeds (tabular RL is noisy) and plot mean ± standard error
curves = utils.seeded_curves(norvig_lambda_specs, "NorvigGrid-v0", gamma=gamma, n_runs=n_runs, n_seeds=n_seeds)
title = f"Avg. reward on NorvigGrid-v0 with SARSA(λ) (mean ± se, {n_seeds} seeds)"
utils.plot_learning_curves(curves, title=title, yaxis_title="Avg. reward", band="se")

<details><summary>Solution</summary>

```python
@dataclass
class TD_LambdaConfig(AgentConfig):
    lambda_: float = 0.95


defaultTDLambdaConfig = TD_LambdaConfig()


class SARSA_lambda(SARSA):
    def __init__(
        self,
        env: DiscreteEnviroGym,
        config: TD_LambdaConfig = defaultTDLambdaConfig,
        gamma: float = 0.99,
        seed: int = 0,
    ):
        super().__init__(env, config, gamma, seed)
        self.lambda_ = config.lambda_
        self.e = np.zeros((self.num_states, self.num_actions), dtype=np.float32)

    def run_episode(self, seed) -> list[float]:
        self.e[:, :] = 0
        return super().run_episode(seed)

    def observe(self, exp: Experience):
        s_t, a_t, r_t, s_tp1, a_tp1 = exp.obs, exp.act, exp.reward, exp.new_obs, exp.new_act
        delta_t = r_t + self.gamma * self.Q[s_tp1, a_tp1] - self.Q[s_t, a_t]
        self.e[s_t, a_t] += 1

        self.Q += self.config.lr * delta_t * self.e  # broadcast update
        self.e *= self.gamma * self.lambda_
```
</details>

### When do eligibility traces actually help?

Eligibility traces speed up **credit assignment**: a 1-step method moves the reward signal only about *one step* closer to the start per episode, whereas SARSA($\lambda$) credits the whole recent trajectory in one update. On a tiny grid the reward only has to travel a few steps, so a 1-step method keeps up just fine. Eligibility traces really shine when **rewards are delayed and paths are long**.

To see this, here's a larger 8×8 gridworld with a *sparse* reward (+1 only at the goal in the top-right, 0 everywhere else), so an optimal path is 14 steps long and the reward has a long way to propagate to the starting state. We compare 1-step SARSA against SARSA($\lambda$), averaging over a few seeds since tabular RL is noisy:

In [19]:
gym.envs.registration.register(
    id="LargeGrid-v0",
    entry_point=DiscreteEnviroGym,
    max_episode_steps=200,
    nondeterministic=False,
    kwargs={"env": GridWorld(".......G\n........\n........\n........\n........\n........\n........\nS.......")},
)

gamma = 0.99
n_runs = 150
n_seeds = 5
agent_specs = [
    (SARSA, AgentConfig(epsilon=0.2, lr=0.2), "SARSA (1-step)", (99, 110, 250)),
    (SARSA_lambda, TD_LambdaConfig(epsilon=0.2, lr=0.2, lambda_=0.8), "SARSA(λ=0.8)", (239, 85, 59)),
]


# average over several seeds (tabular RL is noisy) and plot mean ± se, reusing each spec's colour
curves = utils.seeded_curves([(Cls, cfg, name) for Cls, cfg, name, _ in agent_specs],
                             "LargeGrid-v0", gamma=gamma, n_runs=n_runs, n_seeds=n_seeds)
title = f"Eligibility traces learn faster on a larger gridworld (8×8, sparse reward; mean ± se, {n_seeds} seeds)"
utils.plot_learning_curves(curves, title=title, band="se", colors=[color for *_, color in agent_specs])

SARSA($\lambda$) climbs noticeably faster: On the tiny Norvig grid this gap was negligible; but it grows with the length of the credit-assignment chain, which is why traces become important in environments with longer rollouts (and why we use their cousin, GAE, in PPO later on).

A caveat worth knowing: the trace here is an *accumulating* trace, and with a large $\lambda$ (say 0.95) and a high learning rate, the repeated updates along a long, meandering exploratory episode can become unstable and actually *hurt* performance. $\lambda$ is a genuine bias-variance knob - $\lambda$ = 0 recovers 1-step TD (low variance, slow propagation) and $\lambda$ → 1 approaches Monte-Carlo (fast propagation, high variance) - and moderate values around 0.8–0.9 are usually a safe default.

## Q-Learning v.s SARSA

`gym` provides a large set of environments with which to test agents against. We can see all available environments via the dictionary `gym.envs.registry` (keys are env names, values are `EnvSpec` objects describing how to build each env).

Have a look at [the gymnasium library](https://gymnasium.farama.org/) for descriptions of these environments. As written, our SARSA and Q-Learning agents will only work with environments that have both discrete observation and discrete action spaces.

We'll modify the above code to use environment `gym.make("CliffWalking-v1", is_slippery=False)` instead (see [this link](https://gymnasium.farama.org/environments/toy_text/cliff_walking/)). This is the classic deterministic cliff-walking gridworld from Sutton & Barto. (Gymnasium 1.x renamed it from `CliffWalking-v0` to `CliffWalking-v1` when it added an optional slippery mode; slippery is off by default, and we pass `is_slippery=False` just to make that explicit. Otherwise it's the same environment as `CliffWalking-v0`.) We have the following graph from Sutton & Barto, Example 6.6, that displays the sum of reward obtained for each episode, as well as the policies obtained (SARSA takes the safer path, Q-Learning takes the optimal path). You may want to check out [this post](https://towardsdatascience.com/walking-off-the-cliff-with-off-policy-reinforcement-learning-7fdbcdfe31ff).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/cliff_pi.png" width="450"><br>
<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/cliff.png" width="450">

Do you get a similar result when you run the code below?

Some notes:

* Use $\gamma = 1$ as described in Sutton & Barto, Example 6.6.
* Try tweaking the learning rate and epsilon (start with $\epsilon = 0.1$) to try and cause SARSA to take the cautious path, while Q-Learning takes the risky path.
* We've included some helper functions to display the value of each state, as well as the policy an agent would take, given the Q-value function.
* One of the bonus exercises we've suggested is to write your own version of `CliffWalking-v1` by writing a class similar to the `Norvig` class you have been working with. If you do this correctly, then you'll also be able to make a cheating agent.
* We've defined a `cliffwalk_imshow` helper function for you, which visualises your agent's path (and reward at each square).


<!-- I don't like this detail. I think it's confusing. 
<details>
<summary>Question - why do you think it's okay to use gamma=1 here?</summary>

There exists no policy that can milk infinite reward out of the environment, and Q-learning learns online, so there's no worry of a strange policy wandering around the environment forever. 
</details>
-->

In [20]:
gamma = 1
n_runs = 500
config_cliff = AgentConfig(epsilon=0.1, lr=0.1, optimism=0)
cliff_specs = [(QLearning, config_cliff, "Q-Learning"), (SARSA, config_cliff, "SARSA")]

# train one agent of each type (kept in `agents` for the greedy-rollout section below), show its
# learned policy, and collect its cumulative-mean return. The curves are near-identical across seeds
# here, so a single run is plenty - no need to average.
agents = []
returns_list = []
for AgentCls, config, name in cliff_specs:
    agent = AgentCls(gym.make("CliffWalking-v1", is_slippery=False), config, gamma, seed=0)
    returns = agent.train(n_runs)
    agents.append(agent)
    returns_list.append(utils.cummean(returns))
    V = agent.Q.max(axis=-1).reshape(4, 12)
    pi = agent.Q.argmax(axis=-1).reshape(4, 12)
    cliffwalk_imshow(V, pi, title=f"CliffWalking: {name} Agent", width=800, height=400)

# rewards are always negative; plot the magnitude on a reversed log y-axis with a "-" tick prefix,
# so the axis reads as negative reward (closer to 0 = better = higher)
fig = line(
    [-r for r in returns_list],
    names=[name for _, _, name in cliff_specs],
    template="simple_white",
    title="Q-Learning vs SARSA on CliffWalking-v1",
    labels={"x": "Episode", "y": "Avg. reward", "variable": "Agent"},
    log_y=True,
    width=700,
    height=400,
    return_fig=True,
)
fig.update_layout(yaxis=dict(autorange="reversed", tickprefix="-"))
fig.show()

### Online reward vs. the final greedy policy

The plot above measures reward **while the agents are still learning** - i.e. while they're still acting $\epsilon$-greedily and so still occasionally stepping off the cliff. On that metric SARSA wins, and the reason is exactly that it's **on-policy**: SARSA learns the value of the policy it is *actually following* (the $\epsilon$-greedy one), so it "knows" that walking along the cliff edge is dangerous *because exploration will sometimes push it off*, and it learns the safe detour.
 
Q-learning is **off-policy**: it learns the value of the *optimal greedy* policy regardless of how it's behaving, so it learns to hug the cliff edge - and then its own $\epsilon$-exploration keeps knocking it into the cliff (−100 each time), tanking its online reward.

But objectively Q-learning has learned the optimal policy, whereas SARSA has not. We can see this by measuring the return when exploration is turned off.

In [21]:
def greedy_return(env: gym.Env, Q: Arr, max_steps: int = 200) -> float:
    """Roll out the greedy (epsilon=0) policy implied by Q, and return its total reward."""
    obs, info = env.reset()
    total_reward = 0.0
    for _ in range(max_steps):
        obs, reward, terminated, truncated, info = env.step(int(Q[obs].argmax()))
        total_reward += reward
        if terminated or truncated:
            break
    return total_reward


for agent in agents:
    ret = greedy_return(gym.make("CliffWalking-v1", is_slippery=False), agent.Q)
    print(f"{agent.name:10s} greedy-policy return = {ret:.0f}")

QLearning  greedy-policy return = -13
SARSA      greedy-policy return = -15


Now the ranking **flips**: Q-learning's greedy policy takes the short optimal path along the cliff edge (return $\approx -13$), while SARSA's greedy policy keeps to the longer safe detour (return $\approx -15$).

So which is better? It depends what you care about. 

- If you care about reward **during learning**, or you'll keep exploring/acting $\epsilon$-greedily at deployment (e.g. safety matters), **SARSA** is better - it accounts for the cost of its own exploration.
- If you only care about the **final policy you'll deploy greedily**, **Q-learning** is better - it directly targets the optimal policy.

This is the key difference between on- and off-policy control: SARSA is aware of the policy it is learning from and acts accordingly, whereas Q-learning is only trying to glean what the optimal policy is, with no concern for the performance of the policy while it is learning.

## Bonus - Q($\lambda$)

We can take the same approach for Q-Learning too. However, we need to be careful: the eligibility trace propagates forward updates from the past, and whenever an exploratory action was taken, we are now doing updates off-policy, which is not what we want. One remedy is as follows:

**Watkins' $Q(\lambda)$**: Every time we take an exploratory action that was not the greedy action, we reset the eligibility traces of all the earlier state-action pairs to zero. The pair we just visited still gets a trace of 1, since its own 1-step Q-learning update is valid off-policy (this is the version in Sutton & Barto, 1st edition, section 7.6).
$$
\begin{align*}
e_0(s,a) &= 0 \\
e_t(s,a) &= \begin{cases}
    \mathbf{1}(s = s_t, a = a_t) & \text{if } a_t \neq a^*_t & \text{ (explore) }\\ 
    \gamma \lambda e_{t-1}(s,a) + 1 & \text{if } (s,a) = (s_t,a_t) \text{ and } a_t = a^*_t & \text{ (visit + exploit)}\\ 
    \gamma \lambda e_{t-1}(s,a) & \text{if } (s,a) \neq (s_t,a_t) \text{ and } a_t = a^*_t &\text{ (no visit + exploit)}\\ 
\end{cases} \\
\text{ where } a_t^* &= \arg\max_{a} Q(s_t, a)
\end{align*}
$$

This "cuts" the contribution if an exploratory action was taken instead of the greedy action. However, if the agent is very often taking exploratory actions,
then rarely will TD-backups of more than a few steps happen, defeating the entire point of using eligibility traces.

We leave it to you to implement this if you wish.

## Bonus - build your own CliffWalking environment

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 30-45 minutes on this exercise, if you choose to do it.
> ```

You should return to this exercise at the end if you want to, or continue with the bonus material below.

You can define your own version of `CliffWalking-v1` by subclassing `Environment` directly and writing your own `__init__` and `dynamics` - the same interface the `Toy` environment above uses. (`GridWorld` already understands `'C'` cliff cells, so you could also do this in a handful of lines just by writing the right ASCII map.) You can do this without guidance, or you can get some more guidance from the hint below.

Some notes for this task:

* The random agent will take a *very* long time to accidentally stumble into the goal state - it averages a return of about $-935$, and on its own takes longer to run than all the other agents put together. The solution below drops it, but feel free to put it back in.
* As soon as you hit the cliff, you should immediately move back to the start square, i.e. in pseudocode:
    ```python
    if new_state in cliff:
        new_state = start_state
        reward = -100
    ```
    This means you'll never calculate Q from the cliff, so your Q-values will always be zero here.


<details>
<summary>Hints (for both methods)</summary>

The main way in which the `CliffWalking` environment differs from the `Norvig` gridworld is that the former has cliffs while the latter has walls. Cliffs and walls have different behaviour; you can see how the cliffs affect the agent by visiting the documentation page for `CliffWalking-v1`.

#### `__init__`

This mainly just involves setting the dimensions of the space, the position of the start and terminal states, and parameters like `penalty`. Rather than walls, you'll need to define the position of the **cliffs** (which behave differently).

#### `dynamics`

Your `dynamics` should differ from the Norvig gridworld's in two ways:

* No slippage probability, each action is deterministic (although it would be interesting to experiment with this and see what effect it has!)
* Instead of walls, implement cliffs: "when you walk into a cliff, you get a reward of -100 and go back to the start state". Walking off the edge of the grid should still leave you where you are.

</details>

In [22]:
class CliffWalking(Environment):
    def __init__(self, penalty=-1):
        self.height = 4
        self.width = 12
        self.penalty = penalty
        num_states = self.height * self.width
        num_actions = 4
        self.states = np.array([[x, y] for y in range(self.height) for x in range(self.width)])
        self.actions = np.array([[0, -1], [1, 0], [0, 1], [-1, 0]])  # up, right, down, left
        self.dim = (self.height, self.width)

        # start in the bottom-left corner, goal (terminal) in the bottom-right, the cliff in between. As in gymnasium's
        # `CliffWalking-v1`, every step gives `penalty` (including the step into the goal), and the cliff gives -100
        start = 36
        terminal = np.array([47], dtype=int)
        self.cliff = np.arange(37, 47, dtype=int)
        self.goal_rewards = np.array([float(penalty)])

        super().__init__(num_states, num_actions, start=start, terminal=terminal)

    def dynamics(self, state: int, action: int) -> tuple[Arr, Arr, Arr]:
        """
        Returns tuple of (out_states, out_rewards, out_probs) for this given (state, action) pair.
        """

        def state_index(state):
            assert 0 <= state[0] < self.width and 0 <= state[1] < self.height, f"position {state} is off the grid"
            return state[0] + state[1] * self.width

        pos = self.states[state]

        if state in self.terminal:
            return (np.array([state]), np.array([0]), np.array([1]))

        # No slipping; each action is deterministic
        out_probs = np.zeros(self.num_actions)
        out_probs[action] = 1

        out_states = np.zeros(self.num_actions, dtype=int)
        out_rewards = np.zeros(self.num_actions) + self.penalty
        new_states = [pos + x for x in self.actions]

        for i, s_new in enumerate(new_states):
            if not (0 <= s_new[0] < self.width and 0 <= s_new[1] < self.height):
                out_states[i] = state
                continue

            new_state = state_index(s_new)

            # Check if would hit the cliff, if so then get -100 penalty and go back to start
            if new_state in self.cliff:
                out_states[i] = self.start
                out_rewards[i] = -100

            else:
                out_states[i] = new_state

            for idx in range(len(self.terminal)):
                if new_state == self.terminal[idx]:
                    out_rewards[i] = self.goal_rewards[idx]

        return (out_states, out_rewards, out_probs)

    @staticmethod
    def plot_q(Q: Arr, name: str):
        V = Q.max(axis=-1).reshape(4, 12)
        pi = Q.argmax(axis=-1).reshape(4, 12)
        cliffwalk_imshow(V, pi, title=f"CliffWalking: {name} Agent")


gym.envs.registration.register(
    id="CliffWalking-myversion",
    entry_point=DiscreteEnviroGym,
    max_episode_steps=200,
    nondeterministic=True,
    kwargs={"env": CliffWalking(penalty=-1)},
)
gamma = 0.99
seed = 0
config_cliff = AgentConfig(epsilon=0.1, lr=0.1, optimism=0)
env = gym.make("CliffWalking-myversion")
n_runs = 500
args_cliff = (env, config_cliff, gamma, seed)

agents = [Cheater(*args_cliff), QLearning(*args_cliff), SARSA(*args_cliff)]  # , Random(*args_cliff)]
returns_list = []
name_list = []

for agent in agents:
    returns = agent.train(n_runs)
    returns_list.append(utils.cummean(returns))
    name_list.append(agent.name)

line(
    returns_list,
    names=name_list,
    template="simple_white",
    title="Agent performance on CliffWalking-myversion",
    labels={"x": "Episode", "y": "Avg. reward", "variable": "Agent"},
    width=700,
    height=400,
)

<details><summary>Solution</summary>

```python
class CliffWalking(Environment):
    def __init__(self, penalty=-1):
        self.height = 4
        self.width = 12
        self.penalty = penalty
        num_states = self.height * self.width
        num_actions = 4
        self.states = np.array([[x, y] for y in range(self.height) for x in range(self.width)])
        self.actions = np.array([[0, -1], [1, 0], [0, 1], [-1, 0]])  # up, right, down, left
        self.dim = (self.height, self.width)

        # start in the bottom-left corner, goal (terminal) in the bottom-right, the cliff in between. As in gymnasium's
        # `CliffWalking-v1`, every step gives `penalty` (including the step into the goal), and the cliff gives -100
        start = 36
        terminal = np.array([47], dtype=int)
        self.cliff = np.arange(37, 47, dtype=int)
        self.goal_rewards = np.array([float(penalty)])

        super().__init__(num_states, num_actions, start=start, terminal=terminal)

    def dynamics(self, state: int, action: int) -> tuple[Arr, Arr, Arr]:
        """
        Returns tuple of (out_states, out_rewards, out_probs) for this given (state, action) pair.
        """

        def state_index(state):
            assert 0 <= state[0] < self.width and 0 <= state[1] < self.height, f"position {state} is off the grid"
            return state[0] + state[1] * self.width

        pos = self.states[state]

        if state in self.terminal:
            return (np.array([state]), np.array([0]), np.array([1]))

        # No slipping; each action is deterministic
        out_probs = np.zeros(self.num_actions)
        out_probs[action] = 1

        out_states = np.zeros(self.num_actions, dtype=int)
        out_rewards = np.zeros(self.num_actions) + self.penalty
        new_states = [pos + x for x in self.actions]

        for i, s_new in enumerate(new_states):
            if not (0 <= s_new[0] < self.width and 0 <= s_new[1] < self.height):
                out_states[i] = state
                continue

            new_state = state_index(s_new)

            # Check if would hit the cliff, if so then get -100 penalty and go back to start
            if new_state in self.cliff:
                out_states[i] = self.start
                out_rewards[i] = -100

            else:
                out_states[i] = new_state

            for idx in range(len(self.terminal)):
                if new_state == self.terminal[idx]:
                    out_rewards[i] = self.goal_rewards[idx]

        return (out_states, out_rewards, out_probs)

    @staticmethod
    def plot_q(Q: Arr, name: str):
        V = Q.max(axis=-1).reshape(4, 12)
        pi = Q.argmax(axis=-1).reshape(4, 12)
        cliffwalk_imshow(V, pi, title=f"CliffWalking: {name} Agent")
```
</details>

## Bonus - other suggestions

### Exploration rate schedule

Try using a schedule for the exploration rate $\epsilon$ (Large values early to encourage exploration, low values later once the agent has sufficient statistics to play optimally).

Would Q-Learning or SARSA be better off with a scheduled exploration rate?

The Sutton book mentions that if $\epsilon$ is gradually reduced, both methods asymptotically converge to the optimal policy. Is this what you find?

### Other environments

Try other environments like [Frozen Lake](https://gymnasium.farama.org/environments/toy_text/frozen_lake/) and [BlackJack](https://gymnasium.farama.org/environments/toy_text/blackjack/) (source code [here](https://github.com/Farama-Foundation/Gymnasium/blob/main/gymnasium/envs/toy_text/frozen_lake.py), [here](https://github.com/Farama-Foundation/Gymnasium/blob/main/gymnasium/envs/toy_text/blackjack.py)). Note that BlackJack uses `Tuple(Discrete(32), Discrete(11), Discrete(2))` as its observation space, so you will have to write some glue code to convert this back and forth between an observation space of `Discrete(32 * 11 * 2)` to work with our agents as written.

### Double-Q learning

Read Sutton and Barto Section 6.7 Maximisation Bias and Double Learning. Implement Double-Q learning, and compare its performance against SARSA and Q-Learning.

# 3️⃣ Multi-Armed Bandit

> ##### Learning Objectives
>
> * Compare *Upper Confidence Bound* (UCB) methods to other exploration methods.

This material used to be core, but we've moved it to the bonus section, as nothing later on depends on it. Feel free to skip it, and look through the optional readings above if you prefer.

## The environment in brief

The `MultiArmedBandit` environment below is the simplest possible RL environment:

- **No states.** There is only one observation (always `0`), so the agent has nothing to condition on. The episode never terminates.
- **Actions = arms.** Each action is the index of an arm to pull (`0, ..., num_arms-1`).
- **Rewards.** Pulling arm `a` gives a reward sampled from a normal distribution with mean `arm_reward_means[a]` and standard deviation 1. The means are unknown to the agent, so it has to estimate them from the rewards it sees.
- **Stationary or not.** If `stationary=True`, the means are drawn once at `reset` (from $N(0, 1)$) and never change. If `stationary=False`, they all start at 0 and take a small random-walk step every time `step` is called, so the best arm can change over time.
- **Goal.** Maximise total reward, which means balancing *exploration* (trying arms to learn their means) against *exploitation* (pulling the arm that currently looks best).

## Readings

* [Sutton and Barto](https://www.andrew.cmu.edu/course/10-703/textbook/BartoSutton.pdf), Chapter 2 (25 minutes)
  - Section 2.1: A k-armed Bandit Problem through to Section 2.7 Upper-Confidence-Bound Action Selection
  - We won't cover Gradient Bandits. Don't worry about these.
  - Don't worry too much about all the math for the moment if you can't follow it. The earlier sections are more important.

### The `info` dictionary

The environment's `step` method returns five values: `obs`, `reward`, `terminated`, `truncated`, and the `info` dictionary.

`info` can contain anything extra that doesn't fit into the uniform interface - it's up to the environment what to put into it. A good use of this is for debugging information that the agent isn't "supposed" to see. In this case, we'll return the index of the actual best arm. This would allow us to measure how often the agent chooses the best arm, but it would also allow us to build a "cheating" agent that peeks at this information to make its decision.

Cheating agents are helpful because we know that they should obtain the maximum possible rewards; if they aren't, then there's a bug.

### The `render()` method

Render is only used for debugging or entertainment, and what it does is up to the environment. It might render a little popup window showing the Atari game, or it might give you a RGB image tensor, or just some ASCII text describing what's happening. In this case, we'll just make a little plot showing the distribution of rewards for each arm of the bandit.

### Observation and Action Types

A `gym.Env` is a generic type: both the type of the observations and the type of the actions depends on the specifics of the environment.

We're only dealing with the simplest case: a discrete set of actions which are the same in every state (and ditto for observations). In general, the actions could be continuous, or depend on the state.

In [23]:
ObsType = int
ActType = int


class MultiArmedBandit(gym.Env):
    """
    A class representing a multi-armed bandit environment, based on OpenAI Gym's Env class.

    Attributes:
        action_space (gym.spaces.Discrete): The space of possible actions, representing the arms of
            the bandit.
        observation_space (gym.spaces.Discrete): The space of possible observations.
        num_arms (int): The number of arms in the bandit.
        stationary (bool): Indicates whether the reward distribution (i.e. the arm_reward_means) is
            stationary or not.
        arm_reward_means (np.ndarray): The mean rewards for each arm.
    """

    action_space: gym.spaces.Discrete
    observation_space: gym.spaces.Discrete
    num_arms: int
    stationary: bool
    arm_reward_means: np.ndarray

    def __init__(self, num_arms=10, stationary=True):
        """
        Initializes the MultiArmedBandit environment.

        Args:
            num_arms (int): The number of arms for the bandit. Defaults to 10.
            stationary (bool): Whether the bandit has a stationary reward distribution.
        """
        super().__init__()
        self.num_arms = num_arms
        self.stationary = stationary
        self.observation_space = gym.spaces.Discrete(1)
        self.action_space = gym.spaces.Discrete(num_arms)
        self.reset()

    def step(self, arm: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        """
        Takes an action by choosing an arm and returns the result of the action.

        Args:
            arm (ActType): The selected arm to pull in the bandit.

        Returns:
            obs (ObsType): The observation.
            reward (float): The reward.
            terminated (bool): Whether episode has terminated, i.e. for non-timeout related reasons.
            truncated (bool): Whether episode has timed out.
            info (dict): Additional information.
        """
        assert self.action_space.contains(arm)
        if not self.stationary:
            q_drift = self.np_random.normal(loc=0.0, scale=0.01, size=self.num_arms)
            self.arm_reward_means += q_drift
            self.best_arm = int(np.argmax(self.arm_reward_means))
        reward = self.np_random.normal(loc=self.arm_reward_means[arm], scale=1.0)
        obs = 0
        terminated = False
        truncated = False
        info = dict(best_arm=self.best_arm)
        return (obs, reward, terminated, truncated, info)

    def reset(self, seed: int | None = None, options=None) -> tuple[ObsType, dict]:
        """
        Resets the environment to its initial state.

        Args:
            seed (int | None): The seed for random number generation. Defaults to None.
            options (dict): Additional options for environment reset. Defaults to None.

        Returns:
            obs (ObsType): The initial observation.
            info (dict): Additional information.
        """
        super().reset(seed=seed)
        if self.stationary:
            self.arm_reward_means = self.np_random.normal(loc=0.0, scale=1.0, size=self.num_arms)
        else:
            self.arm_reward_means = np.zeros(shape=[self.num_arms])
        self.best_arm = int(np.argmax(self.arm_reward_means))

        obs = 0
        info = {}
        return obs, info

    def render(self, mode="human"):
        """
        Renders the state of the environment, in the form of a violin plot.
        """
        assert mode == "human", f"Mode {mode} not supported!"
        bandit_samples = []
        for arm in range(self.action_space.n):
            bandit_samples += [self.np_random.normal(loc=self.arm_reward_means[arm], scale=1.0, size=1000)]
        plt.violinplot(bandit_samples, showmeans=True)
        plt.xlabel("Bandit Arm")
        plt.ylabel("Reward Distribution")
        plt.show()

### Registering an Environment

Usually we won't use a class like `MultiArmedBandit` directly, because we'll want to wrap it in classes that add various other features that help it work with the rest of the gymnasium library. The `register` function does this for us: it stores information about our `Env` in a registry so that a later call to `gym.make` can look it up using the `id` string that is passed in, and then create an instance of our class with a bunch of extra wrapper classes.

By convention, the `id` strings have a suffix with a version number. There can be multiple versions of the "same" environment with different parameters, and benchmarks should always report the version number for a fair comparison. For instance, `id="ArmedBanditTestbed-v0"` below.

#### `TimeLimit` wrapper

One important wrapper class is the `TimeLimit` wrapper - this is used to ensure that the environment terminates after a fixed number of steps. It works by having the `truncated` flag returned by the `step` method be overridden after `max_episode_steps`, and set to `True`. You can see from looking at the `step` method above that without this, the agent might keep playing forever, because it always returns `terminated=truncated=False`. The time limit is also an essential part of the problem definition: if it were larger or shorter, there would be more or less time to explore, which means that different algorithms (or at least different hyperparameters) would then have improved performance.

A list of the other wrappers you'll see printed below (these are less important):

- `OrderEnforcing` ensures the environment is reset before we start using it
- `PassiveEnvChecker` surrounds the step, reset and render functions to check they follow Gymnasium’s API

In [24]:
gym.envs.registration.register(
    id="ArmedBanditTestbed-v0",
    entry_point=MultiArmedBandit,
    max_episode_steps=max_episode_steps,
    nondeterministic=True,
    reward_threshold=1.0,
    kwargs={"num_arms": 10, "stationary": True},
)

env = gym.make("ArmedBanditTestbed-v0")
print(f"Our env inside its wrappers looks like: {env}")

Our env inside its wrappers looks like: <TimeLimit<OrderEnforcing<PassiveEnvChecker<MultiArmedBandit<ArmedBanditTestbed-v0>>>>>


#### A Note on (pseudo) RNGs

The PRNG that `gym.Env` provides as `self.np_random` is from the [PCG family](https://www.pcg-random.org/index.html). In RL code, you often need massive quantities of pseudorandomly generated numbers, so it's important to have a generator that is both very fast and has good quality output.

When you call `np.random.randint` or similar, you're using the old-school Mersenne Twister algorithm which is both slower and has inferior quality output to PCG. Since Numpy 1.17, you can use `np.random.default_rng()` to get a PCG generator and then use its `integers` method to get random integers.

One trap to watch for: `env.reset(seed=seed)` sets `env.np_random` to exactly `np.random.default_rng(seed)`. If the agent also seeds its own generator with `np.random.default_rng(seed)`, the agent and the environment draw *the same sequence of numbers*, so (for example) the agent's exploration coin flips are correlated with the environment's random outcomes. That's why our agents' `reset` methods derive a separate child stream with `np.random.SeedSequence(seed).spawn(1)[0]`.

## Agent class

Now, you'll start defining agents to act within this class.

You'll be defining your agents as subclasses of the base class `Agent` for agents in a multi-armed bandit environment, given below. It contains the following methods:

* `get_action`, which returns the agent's next action. This doesn't take state as an argument, because the state of the multi armed bandit game is always the same.
    * This isn't implemented for `Agent`, because we'll implement it separately for each agent we build on top of the base agent.
* `observe`, which determines what the model does when it observes a particular action & reward combination.
    * The default behaviour is to do nothing, although for some agents we'll implement more complicated behaviour.
* `reset`, which is run before each episode starts.
    * All that the base method does is reset the random number generator, but again future agents might need to do more, e.g. reset data gathered from the `observe` method.

Additionally, the `run_episode` function will run a single episode of interaction between an agent and the environment. As discussed above, this will terminate after `max_episode_steps` for our environment (we've set this to be 1000). The `run_agent` function calls `run_episode` multiple times in series.

In [25]:
class Agent:
    """
    Base class for agents in a multi-armed bandit environment

    (you do not need to add any implementation here)
    """

    rng: np.random.Generator

    def __init__(self, num_arms: int, seed: int):
        self.num_arms = num_arms
        self.reset(seed)

    def get_action(self) -> ActType:
        raise NotImplementedError()

    def observe(self, action: ActType, reward: float, info: dict) -> None:
        pass

    def reset(self, seed: int) -> None:
        # The env is seeded with the same `seed`, and gymnasium's `env.np_random` is exactly `np.random.default_rng(seed)`.
        # Seeding our generator the same way would make the agent's random draws the very same numbers as the env's, so
        # we use a separate child stream.
        self.rng = np.random.default_rng(np.random.SeedSequence(seed).spawn(1)[0])


def run_episode(env: gym.Env, agent: Agent, seed: int) -> tuple[Arr, Arr]:
    """
    Runs a single episode of interaction between an agent and an environment.

    Args:
        env (gym.Env): The environment in which the agent operates.
        agent (Agent): The agent that takes actions in the environment.
        seed (int): The seed for random number generation to ensure reproducibility.

    Returns:
        A tuple containing arrays of rewards received in each step and a flag indicating if the
        chosen arm was best.
    """
    (rewards, was_best) = ([], [])

    env.reset(seed=seed)
    agent.reset(seed=seed)

    done = False
    while not done:
        arm = agent.get_action()
        obs, reward, terminated, truncated, info = env.step(arm)
        done = terminated or truncated
        agent.observe(arm, reward, info)
        rewards.append(reward)
        was_best.append(1 if arm == info["best_arm"] else 0)

    rewards = np.array(rewards, dtype=float)
    was_best = np.array(was_best, dtype=int)
    return (rewards, was_best)


def run_agent(env: gym.Env, agent: Agent, n_runs=200, base_seed=1) -> tuple[Arr, Arr]:
    all_rewards = []
    all_was_bests = []
    base_rng = np.random.default_rng(base_seed)
    for n in tqdm(range(n_runs)):
        seed = base_rng.integers(low=0, high=10_000, size=1).item()
        (rewards, corrects) = run_episode(env, agent, seed)
        all_rewards.append(rewards)
        all_was_bests.append(corrects)
    return np.array(all_rewards), np.array(all_was_bests)

### Exercise - implement `RandomAgent`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

You should fill in the `get_action` method from `RandomAgent`, which is a subclass of `Agent`. This agent should pick an arm at random, i.e. from the range `[0, ..., num_arms-1]`, and return it as an integer. You won't need to change the `observe` method since the random agent doesn't change its behaviour based on observations, and you won't need to change the `reset` method since there are no observations which need to be reset.

This agent will be useful as a baseline to ensure the environment has no bugs. If your later agents are doing worse than random, you have a bug! Later, we'll add some smarter agents.

When you've finished implementing this agent, running the cell below will:
        
- Verify that `RandomAgent` pulls the optimal arm with frequency roughly `1/num_arms`.
- Verify that the average reward is very roughly zero. This is the case since the mean reward for each arm is centered on zero.
- Verify that every arm can be chosen.

In [26]:
class RandomAgent(Agent):
    def get_action(self) -> ActType:
        return self.rng.integers(low=0, high=self.num_arms)

    def __repr__(self):
        return "RandomAgent"


tests.test_random_agent(RandomAgent)

  0%|          | 0/200 [00:00<?, ?it/s]

Expected correct freq: 0.1, actual: 0.099545
Expected average reward: 0.0, actual: 0.005870
All tests in `test_random_agent` passed!


<details><summary>Solution</summary>

```python
class RandomAgent(Agent):
    def get_action(self) -> ActType:
        return self.rng.integers(low=0, high=self.num_arms)

    def __repr__(self):
        return "RandomAgent"
```
</details>

### Exercise - Implement reward averaging
> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 20-30 minutes on this exercise.
> ```

You should now complete the methods for the `RewardAveraging` agent, which applies the reward averaging algorithm as detailed in Sutton and Barto section 2.4, "Incremental Implementation":

* You should fill in `observe` to keep track of the number of times $n$ each arm has been pulled, as well as the value $Q_n$ for each arm.
    * Gotcha - in S & B notation, $n$ is the number of times this particular arm has been pulled, not the total number of actions taken!
    * We recommend defining arrays `N` and `Q`, each of length `num_arms`, to keep track of all these values.
* You should fill in `get_action` with an epsilon-greedy method: taking a random action with probability `epsilon`, and taking the best action based on the current value of $Q$ with probability `1-epsilon` (see Sutton & Barto).
* You should fill in `reset` to call the reset method from the parent class, *and* make sure that the tracked values of $(n, Q_n)$ are set back to appropriate values at the start of each episode.
    * Note, the `reset` method is also called before the very first run, so you don't need to define `N` and `Q` in the init method.
    * The `Q` values should be initialized according to the optimism value of this agent.
    * Ensure the `Q` values are stored as float (and not numpy's default, integer)

<details>
<summary>Hint - average reward formula</summary>

$$Q_k = Q_{k-1} + \frac{1}{k}[R_k - Q_{k-1}]$$

Where $k$ is the number of times the action has been taken, $R_k$ is the reward from the kth time the action was taken, and $Q_{k-1}$ is the average reward from the previous times this action was taken (this notation departs slightly from the S&B notation, but may be more helpful for our implementation).

**Important - $k$ is not the total number of timesteps, it's the total number of times you've taken this particular action.**

**Note on optimism:** While the formula above doesn't explicitly include it, optimism is incorporated by initializing $Q_0 = \text{optimism}$ (rather than 0) for each arm; the formula then applies as written.

</details>

We've given you a function for plotting multiple agents' reward trajectories on the same graph, with an optional moving average parameter to make the graph smoother.

In [27]:
class RewardAveraging(Agent):
    def __init__(self, num_arms: int, seed: int, epsilon: float, optimism: float):
        self.epsilon = epsilon
        self.optimism = optimism
        super().__init__(num_arms, seed)

    def get_action(self):
        if self.rng.random() < self.epsilon:
            return self.rng.integers(low=0, high=self.num_arms).item()
        else:
            return np.argmax(self.Q)

    def observe(self, action, reward, info):
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]

    def reset(self, seed: int):
        super().reset(seed)
        self.N = np.zeros(self.num_arms)
        self.Q = np.full(self.num_arms, self.optimism, dtype=float)

    def __repr__(self):
        # For the legend, when plotting
        return f"RewardAveraging(eps={self.epsilon}, optimism={self.optimism})"


tests.test_reward_averaging(RewardAveraging)

num_arms = 10
stationary = True
names = []
all_rewards = []
env = gym.make("ArmedBanditTestbed-v0", num_arms=num_arms, stationary=stationary)

for optimism in [0, 5]:
    agent = RewardAveraging(num_arms, 0, epsilon=0.01, optimism=optimism)
    (rewards, num_correct) = run_agent(env, agent, n_runs=N_RUNS, base_seed=1)
    all_rewards.append(rewards)
    names.append(str(agent))
    print(agent)
    print(f" -> Frequency of correct arm: {num_correct.mean():.4f}")
    print(f" -> Average reward: {rewards.mean():.4f}")

utils.plot_rewards(all_rewards, names, moving_avg_window=15)

All tests in `test_reward_averaging` passed!


  0%|          | 0/200 [00:00<?, ?it/s]

RewardAveraging(eps=0.01, optimism=0)
 -> Frequency of correct arm: 0.4826
 -> Average reward: 1.2296


  0%|          | 0/200 [00:00<?, ?it/s]

RewardAveraging(eps=0.01, optimism=5)
 -> Frequency of correct arm: 0.7444
 -> Average reward: 1.4737


<details>
<summary>Question - can you interpret these results?</summary>

At the very start, the more optimistic agent performs worse, because it explores more and exploits less (here you will see it at the very beginning if you zoom in). Its estimates are wildly over-optimistic, so even if it finds a good arm, its Q-value for that arm will decrease. On the other hand, if the realistic agent finds a good arm early on, it'll probably return to exploit it.

However, the optimistic agent eventually outperforms the realistic agent, because its increased exploration means it's more likely to converge on the best arm.
</details>

<details>
<summary>Question - how do you think these results would change if epsilon was decreased for both agents?</summary>

You should expect the optimistic agent to outperform the realistic agent even more. The smaller epsilon is, the more necessary optimism is (because without it the agent won't explore enough).
</details>


<details><summary>Solution</summary>

```python
class RewardAveraging(Agent):
    def __init__(self, num_arms: int, seed: int, epsilon: float, optimism: float):
        self.epsilon = epsilon
        self.optimism = optimism
        super().__init__(num_arms, seed)

    def get_action(self):
        if self.rng.random() < self.epsilon:
            return self.rng.integers(low=0, high=self.num_arms).item()
        else:
            return np.argmax(self.Q)

    def observe(self, action, reward, info):
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]

    def reset(self, seed: int):
        super().reset(seed)
        self.N = np.zeros(self.num_arms)
        self.Q = np.full(self.num_arms, self.optimism, dtype=float)

    def __repr__(self):
        # For the legend, when plotting
        return f"RewardAveraging(eps={self.epsilon}, optimism={self.optimism})"
```
</details>

### Exercise - implement cheater agent

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 5-15 minutes on this exercise.
> It's important for you to understand why cheater agents are important for debugging.
> ```

The cheater agent will always choose the best arm. It's a good idea to implement, because you can compare your other agents to it to make sure they're not doing better than the cheater (if they are, you probably have a bug!).

You should fill in the methods `get_action` and `observe` below. The cheater agent will always choose the best arm available to it (remember that the best arm is stored in the `info` dictionary of each observation - see the `step` method in the `MultiArmedBandit` class).

In [28]:
class CheatyMcCheater(Agent):
    def __init__(self, num_arms: int, seed: int):
        super().__init__(num_arms, seed)
        self.best_arm = 0

    def get_action(self):
        return self.best_arm

    def observe(self, action: int, reward: float, info: dict):
        self.best_arm = info["best_arm"]

    def __repr__(self):
        return "Cheater"


tests.test_cheaty_mc_cheater(CheatyMcCheater)

cheater = CheatyMcCheater(num_arms, 0)
reward_averaging = RewardAveraging(num_arms, 0, epsilon=0.1, optimism=0)
random = RandomAgent(num_arms, 0)

names = []
all_rewards = []

for agent in [cheater, reward_averaging, random]:
    (rewards, num_correct) = run_agent(env, agent, n_runs=N_RUNS, base_seed=1)
    names.append(str(agent))
    all_rewards.append(rewards)

utils.plot_rewards(all_rewards, names, moving_avg_window=15)

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

All tests in `test_cheaty_mc_cheater` passed!


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

<details><summary>Solution</summary>

```python
class CheatyMcCheater(Agent):
    def __init__(self, num_arms: int, seed: int):
        super().__init__(num_arms, seed)
        self.best_arm = 0

    def get_action(self):
        return self.best_arm

    def observe(self, action: int, reward: float, info: dict):
        self.best_arm = info["best_arm"]

    def __repr__(self):
        return "Cheater"
```
</details>

### The Authentic RL Experience

It would be nice if we could say something like "optimistic reward averaging is a good/bad feature that improves/decreases performance in bandit problems." Unfortunately, we can't justifiably claim either at this point.

Usually, RL code fails silently, which makes it difficult to be confident that you don't have any bugs. I had a bug in my first attempt that made both versions appear to perform equally, and there were only 13 lines of code, and I had written similar code before.

The number of hyperparameters also grows rapidly, and hyperparameters have interactions with each other. Even in this simple problem, we already have two different ways to encourage exploration (`epsilon` and `optimism`), and it's not clear whether it's better to use one or the other, or both in some combination. It's actually worse than that, because `epsilon` should probably be annealed down at some rate.

Even in this single comparison, we trained 200 agents for each version. Is that a big number or a small number to estimate the effect size? Probably we should like, compute some statistics? And test with a different number of arms - maybe we need more exploration for more arms? The time needed for a rigorous evaluation is going to increase quickly.

As we continue onward to more complicated algorithms, keep an eye out for small discrepancies or minor confusions. Look for opportunities to check and cross-check everything, and be humble with claims.

### Exercise - implement UCB selection
> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> ```

Once you feel good about your `RewardAveraging` implementation, you should implement `UCBActionSelection`.

This should store the same moving average rewards for each action as `RewardAveraging` did, but instead of taking actions using the epsilon-greedy strategy it should use Equation 2.10 in Section 2.7 to select actions using the upper confidence bound.

You should expect to see a small improvement over `RewardAveraging` using this strategy.

*Tip - be careful of division-by-zero errors!*

In [29]:
class UCBActionSelection(Agent):
    def __init__(self, num_arms: int, seed: int, c: float, eps: float = 1e-6):
        super().__init__(num_arms, seed)
        self.c = c
        self.eps = eps

    def get_action(self):
        # This method avoids division by zero errors, and makes sure N=0 entries are chosen by argmax
        ucb = self.Q + self.c * np.sqrt(np.log(self.t) / (self.N + self.eps))
        return np.argmax(ucb)

    def observe(self, action, reward, info):
        self.t += 1
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]

    def reset(self, seed: int):
        super().reset(seed)
        self.t = 1
        self.N = np.zeros(self.num_arms)
        self.Q = np.zeros(self.num_arms)

    def __repr__(self):
        return f"UCB(c={self.c})"


tests.test_ucb(UCBActionSelection)

cheater = CheatyMcCheater(num_arms, 0)
reward_averaging = RewardAveraging(num_arms, 0, epsilon=0.1, optimism=0)
reward_averaging_optimism = RewardAveraging(num_arms, 0, epsilon=0.1, optimism=5)
ucb = UCBActionSelection(num_arms, 0, c=2.0)
random = RandomAgent(num_arms, 0)

names = []
all_rewards = []

for agent in [cheater, reward_averaging, reward_averaging_optimism, ucb, random]:
    (rewards, num_correct) = run_agent(env, agent, n_runs=N_RUNS, base_seed=1)
    names.append(str(agent))
    all_rewards.append(rewards)

utils.plot_rewards(all_rewards, names, moving_avg_window=15)

All tests in `test_ucb` passed!


  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

<details><summary>Solution</summary>

```python
class UCBActionSelection(Agent):
    def __init__(self, num_arms: int, seed: int, c: float, eps: float = 1e-6):
        super().__init__(num_arms, seed)
        self.c = c
        self.eps = eps

    def get_action(self):
        # This method avoids division by zero errors, and makes sure N=0 entries are chosen by argmax
        ucb = self.Q + self.c * np.sqrt(np.log(self.t) / (self.N + self.eps))
        return np.argmax(ucb)

    def observe(self, action, reward, info):
        self.t += 1
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]

    def reset(self, seed: int):
        super().reset(seed)
        self.t = 1
        self.N = np.zeros(self.num_arms)
        self.Q = np.zeros(self.num_arms)

    def __repr__(self):
        return f"UCB(c={self.c})"
```
</details>

## Bonus

Here are a few bonus exercises you can try if you're interested. You could instead progress to the second section, and return to them if you have time.

* Implement the gradient bandit algorithm.
* Implement an environment and an agent for the contextual bandit problem.
* Complete the exercises at the end of Chapter 2 of Sutton and Barto.